# Project scoping canvas

**Space Academy 2026 · Hyperspectral course · Week 2 — Project scoping and use case definition**

Fill this in as a team (1–2 pages). You will present it in the scoping session, and it becomes the first section of
your project notebook (`templates/project_template.ipynb`). Keep it concrete: a reviewer should be able to tell
exactly **what** you will map, **where**, **with which data**, and **how you will know whether it worked**.

---

## 1. Team

| Name | Background | Role in the project |
|------|------------|---------------------|
| Mariolino Daniel III Julabar | Aerospace Engineer | Project lead & primary developer |
| AlReem AlRowahi | Aerospace Engineer | Project design, testing, and analysis support |

## 2. Problem statement (≤ 80 words)

> *Who* has *what* problem *where*, and why does it matter now?

Project Demeter explores vegetation distribution and spectral variability near Massafra, Puglia, Italia, to support agricultural/environmental analysts selecting locations for inspection. A single-date Tanager workflow generates vegetation-index maps and exploratory spectral clusters. The uploaded run produced vegetation candidates and georeferenced outputs; WorldCover provides coarse agreement while ground-truth accuracy remains unknown. Results do not establish cropland, crop health, disease, yield or species.

## 3. Users and decisions

| Question | Your answer |
|----------|-------------|
| Who is the end user (organisation, role)? | Intended agricultural/environmental analysts; no engaged organisation. |
| Which decision will your output support? | Where to inspect vegetation patterns and collect independent references. |
| What do they use today, and why is it not good enough? | Practices not surveyed; project baseline is NDVI. Added hyperspectral accuracy is unproven. |
| How often do they need the information (once, monthly, near-real-time)? | One-date demonstration; repeat processing available, no operational schedule. |
| In what form do they need it (map, number, alert, report, API)? | Maps, statistics, HTML report, COGs, metadata and ZIP; reusable pipeline, not a deployed service. |

## 4. Why hyperspectral?

What can Tanager/EnMAP see that Sentinel-2 or Landsat cannot? Name the **specific spectral features** (wavelength
ranges, absorption features, indices) your method relies on. If multispectral data would work equally well, say so.
That is a valid finding, but then the project must add value in another way.

NDVI (860/665 nm), NDRE (790/720 nm), NDMI (860/1650 nm), PCA and full-spectrum SAM/SID/clustering explore vegetation variability. Multispectral imagery supports the baseline; full-spectrum exploration is the added scope, not a demonstrated accuracy advantage.

## 5. Area of interest and time frame

- AOI (name, bounding box in lon/lat, size in km²): Massafra area, Puglia, Italia; 512 × 512 crop, 30 m, EPSG:32633; projected area 235.9296 km². Approximate enclosing crop bbox (W,S,E,N): (17.059788602, 40.504259480, 17.245570198, 40.645918944); administrative boundary unverified.
- Period of interest: 30 April 2025, 10:17:24.870 UTC (catalogue).
- Why this AOI is representative of the wider problem: Selected mixed vegetation/coastal case study; not established as representative of all Puglia.

## 6. Data availability (use `16_scene_finder.ipynb`)

| Dataset | Scene IDs / product | Date(s) | Cloud % | Role in the project |
|---------|---------------------|---------|---------|---------------------|
| Tanager | `20250430_101724_87_4001`, ortho SR HDF5 | 30 Apr 2025 | 2% scene catalogue metadata; crop QA applied, 98.01% pixels retained | primary analysis |
| EnMAP / Sentinel-2 / Landsat (optional) | No external scene; same-scene Sentinel-2-like bandpass simulation only (not independent validation) | 30 Apr 2025 | N/A | Bandpass comparison, not validation |
| Reference / labels | ESA WorldCover 2021 v200; tile N39E015 | 2021 | N/A | Coarse external agreement, not ground-truth accuracy |
| Ancillary | Catalogue footprint, source CRS/transform | Source-associated | N/A | Geographic checks / export |

- [x] At least one usable Tanager scene covers the AOI (checked with thumbnails and QA masks).
- [ ] If the method needs change detection: at least two dates over the same AOI.
- [x] A source of reference data for validation exists (if not, say how you will validate).

## 7. Method (first version)

1. Preprocessing (bad bands, masks, subsetting) — `11_preprocessing.ipynb`
2. Features (indices, PCA/MNF, continuum removal…) — `13_feature_extraction_dim_reduction.ipynb`
3. Core analysis (target detection / unmixing / classification / regression / change detection) — notebooks 14, 15, 20, 30
4. Output product (map, statistic, alert)

Draw the processing chain as boxes and arrows. A photo of a whiteboard is fine.

[Scene/QA] → [Crop/band screening] → [Indices/PCA] → [NDVI > 0.5 + five clusters] → [WorldCover coarse agreement] → [Sensitivity/report/COGs]

Supplied report run `20261113T101551_705207Z`, generated 2026-11-13 10:17:26 UTC: 256,933 valid pixels (98.01%; 231.2397 km²); 145,771 NDVI vegetation candidates, 13,119.39 ha. Crop rows [127,639), columns [182,694), 30 m, EPSG:32633; projected bounds (W,S,E,N): 674520,4486140,689880,4501500 m. The report records 28 technical checks as PASS (including COG geometry, band descriptions, round-trip arrays and no raw HDF5 in the output package). Clean-kernel execution remains unverified; human figure review remains required. These are completed report results, not a new execution of this documentation update.

## 8. Validation plan (use `21_validation_accuracy_assessment.ipynb`)

| Item | Plan |
|------|------|
| Reference data | ESA WorldCover 2021 v200 through Microsoft Planetary Computer; mode aggregation from 10 m to the 30 m crop. Vegetation codes 10,20,30,40,90,95,100; secondary cropland reference 40. No manual labels planned. |
| Metric(s) | OA agreement, precision/UA, recall/PA, per-class F1/IoU, macro-F1, confusion matrices and conditional block-bootstrap intervals. Reported vegetation OA/F1/IoU: 0.696676/0.787899/0.650027; cropland: 0.430315/0.273993/0.158744. |
| Sampling / split | All 256,933 overlapping valid pixels, including mixed pixels; no training or threshold tuning against reference. No independent train/test split claimed. 169 spatial blocks of 40 × 40 pixels; 500 bootstrap replicates. Conditional vegetation-F1 interval: 0.765342–0.809086; block size provisional. |
| Success criterion | Descriptive vegetation agreement F1 ≥ 0.80 and above constant all-vegetation baseline. Not met: 0.787899 versus baseline 0.926314. NDVI macro-F1 is higher (0.627832 versus 0.463157). Not an operational approval or threshold-tuning rule. |

WorldCover is a coarse external agreement reference, not ground truth. Limitations: temporal/seasonal mismatch (2021/2025), 10 m/30 m aggregation, mixed pixels, reference errors, spatial dependence and different class definitions. Greenness is not cropland; spectral clusters have no one-to-one WorldCover mapping. Conditional intervals exclude reference errors and do not establish true-accuracy or map-area confidence intervals.

## 9. Deliverables

- [ ] Reproducible notebook (runs top to bottom) *(Clean-kernel execution of this final edited file still requires verification.)*
- [x] Output layer(s) as Cloud-Optimised GeoTIFF + metadata *(Written and technical checks passed in the supplied report; GIS alignment review remains.)*
- [x] Validation table *(WorldCover agreement CSVs, confusion matrices and conditional intervals.)*
- [x] Mock-up or prototype of how the user receives the result (dashboard, API, report) *(Standalone HTML report.)*
- [ ] Final presentation (see `templates/presentation_template.md`)

## 10. Risks and fallbacks

| Risk | Likelihood | Impact | Fallback |
|------|-----------|--------|----------|
| No cloud-free scene over the AOI | Residual contamination possible | High | Inspect QA; change crop/scene if needed |
| No reference data | WorldCover available; no manual ground truth | High | Report coarse agreement and reference limitations, not true accuracy |
| Method does not separate the classes | Unassessed | Medium–High | Keep neutral cluster names; do not infer semantic classes from WorldCover agreement |

## 11. Timeline

| When | Milestone |
|------|-----------|
| Thu 19 Nov | This canvas drafted in the scoping session |
| Mon 23 Nov | Canvas due · data downloaded, first quicklooks |
| Tue 24 Nov | Working method on one scene |
| Wed 25 Nov | Build afternoon — final validation numbers |
| Thu 26 Nov | Submission in the morning, presentation in the afternoon |

Progress: latest supplied report `20261113T101551_705207Z` generated 2026-11-13 10:17:26 UTC. Technical outputs passed 28 checks; no independent manual ground truth, clean-kernel confirmation or completed figure review is inferred.

---
*Data: Tanager — CC-BY-4.0 © Planet Labs PBC. Credit every dataset you use.*

*ESA WorldCover 2021 v200, European Space Agency, CC BY 4.0; Microsoft Planetary Computer. Deliverable status is based on the supplied report; presentation and final clean-kernel execution are not verified here.*


# Project Demeter: Hyperspectral Vegetation Mapping of Puglia, Italia
### Space Academy 2026 · Hyperspectral course · Final graduation project

**Team:** Mariolino Daniel III Julabar · AlReem AlRowahi · **Theme:** Agriculture · **Version:** 13 November 2026

Run setup, configure inputs, and execute in order. This notebook follows the course project template's sections 1–10. Validation is deliberately adapted to coarse agreement with ESA WorldCover 2021 v200, not independent ground-truth accuracy. The supplied helper is embedded for reproducibility; no course clone is needed.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mariolinoiii/team_demeter_hyperspectral_project/blob/main/Project_Demeter.ipynb)

*Run the setup cell first to install packages and load the embedded course helper, then execute the remaining cells in order.*


In [ ]:
import sys,subprocess,platform,importlib,importlib.metadata,importlib.util,hashlib
from pathlib import Path
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable,'-m','pip','install','-q','requests','h5py','numpy','matplotlib','pandas','scipy','scikit-learn','joblib','rasterio','rioxarray','xarray','geopandas','shapely'],check=True)
HELPER_SOURCE = '"""\nhsi_utils.py — shared helpers for the Space Academy 2026 hyperspectral course.\n\nEvery course notebook repeats the same handful of steps: find a Tanager scene,\ndownload it, read a few bands, mask bad pixels, drop bad bands, plot spectra and\nexport results. This module collects those steps in one place so the notebooks\ncan focus on the science.\n\nQuick start\n-----------\n>>> import hsi_utils as hu\n>>> scene = hu.get_scene("20250608_091605_90_4001")        # finds or downloads the HDF5\n>>> cube, wl = scene.read_cube(window=(200, 400, 300, 500))  # (bands, rows, cols), nm\n>>> ndvi = hu.nd(scene.read_band(860), scene.read_band(665))\n\nConventions used throughout the course\n--------------------------------------\n* Cubes are numpy arrays shaped ``(bands, rows, cols)`` in float32 reflectance (0–1).\n* Invalid pixels (no-data, cloud, cirrus, fill values) are set to ``NaN``.\n* A *window* is a tuple ``(row_start, row_stop, col_start, col_stop)`` in pixel units.\n* Wavelengths are always in nanometres.\n\nTanager data: CC-BY-4.0 © Planet Labs PBC.\n"""\nfrom __future__ import annotations\n\nimport json\nimport os\nimport re\nfrom pathlib import Path\n\nimport h5py\nimport numpy as np\n\n# --------------------------------------------------------------------------------------\n# Paths and catalogue\n# --------------------------------------------------------------------------------------\nCOURSE_DIR = Path(__file__).resolve().parent\nDATA_DIR = COURSE_DIR / "data"\nLABEL_DIR = DATA_DIR / "labels"\nLIBRARY_DIR = DATA_DIR / "spectral_library"\nOUTPUT_DIR = COURSE_DIR / "outputs"\n\nSTAC_ROOT = "https://www.planet.com/data/stac/tanager-core-imagery"\nCOLLECTIONS = [\n    "agriculture", "coastal-water-bodies", "energy-mining", "fire", "GHG-plumes",\n    "natural-lands", "snow-ice", "urban", "ROCX2025",\n]\n\nH5_ROOT = "HDFEOS/GRIDS/HYP/Data Fields"\n\n# Scenes used in the course notebooks. The theme notebooks (00–06) already use the first six.\nSCENES = {\n    "20241215_185916_33_4001": dict(collection="fire", place="Malibu, California, USA",\n                                    theme="Wildfire — Franklin fire burn scar (Dec 2024)"),\n    "20250123_185518_92_4001": dict(collection="fire", place="Santa Monica Mountains, California, USA",\n                                    theme="Wildfire — after the Palisades fire (Jan 2025)"),\n    "20250219_053251_31_4001": dict(collection="GHG-plumes", place="Korba, Chhattisgarh, India",\n                                    theme="Coal mining, power plants, GHG"),\n    "20250223_165546_32_4001": dict(collection="coastal-water-bodies", place="Usulután, El Salvador",\n                                    theme="Mangroves, estuary, coast, sugar cane"),\n    "20250407_035509_25_4001": dict(collection="urban", place="Ho Chi Minh City, Vietnam",\n                                    theme="Urban fabric, industry, river, vegetation"),\n    "20250511_074311_00_4001": dict(collection="coastal-water-bodies", place="Al Dhafra coast, UAE",\n                                    theme="Shallow coastal water, sabkha, salt ponds"),\n    "20250608_091605_90_4001": dict(collection="agriculture", place="Konya, Türkiye",\n                                    theme="Irrigated agriculture, dry steppe"),\n}\n\n\ndef data_dirs() -> list[Path]:\n    """Folders searched (in order) for Tanager HDF5 files.\n\n    1. ``$TANAGER_DATA_DIR`` (one or more folders separated by ``os.pathsep``)\n    2. ``<course>/data``\n    3. ``<course>/../notebooks`` (where the theme notebooks store their downloads)\n    4. the current working directory\n    """\n    dirs = []\n    env = os.environ.get("TANAGER_DATA_DIR")\n    if env:\n        dirs += [Path(p) for p in env.split(os.pathsep) if p]\n    dirs += [DATA_DIR, DATA_DIR / "subsets"]\n    # Sibling folders that hold the Tanager scenes. The starter-notebook folder has been called\n    # both "notebooks" and "Starting_notebooks", so also pick up any sibling that actually\n    # contains scene files — renaming the folder should not break every notebook.\n    parent = COURSE_DIR.parent\n    dirs += [parent / "notebooks", parent / "Starting_notebooks"]\n    if parent.is_dir():\n        dirs += sorted(d for d in parent.iterdir()\n                       if d.is_dir() and any(d.glob("*_ortho_sr_hdf5.h5")))\n    dirs.append(Path.cwd())\n    seen, out = set(), []\n    for d in dirs:\n        d = d.resolve()\n        if d not in seen:\n            seen.add(d)\n            out.append(d)\n    return out\n\n\ndef download_dir() -> Path:\n    """Where new downloads go: the first ``$TANAGER_DATA_DIR`` entry, else ``<course>/data``."""\n    env = os.environ.get("TANAGER_DATA_DIR")\n    d = Path(env.split(os.pathsep)[0]) if env else DATA_DIR\n    d.mkdir(parents=True, exist_ok=True)\n    return d\n\n\ndef find_local(scene_id: str, product: str = "ortho_sr_hdf5", allow_subset: bool = True) -> Path | None:\n    """Return the path of a locally available HDF5 for ``scene_id``, or ``None``.\n\n    Full scenes are preferred. A subset made with ``tools/make_subsets.py``\n    (``data/subsets/<scene_id>_subset.h5``) is only returned when no full scene exists —\n    note that its pixel coordinates start at (0, 0) of the subset window.\n    """\n    for d in data_dirs():\n        p = d / f"{scene_id}_{product}.h5"\n        if p.exists():\n            return p\n    if allow_subset:\n        for d in data_dirs() + [DATA_DIR / "subsets"]:\n            p = d / f"{scene_id}_subset.h5"\n            if p.exists():\n                return p\n    return None\n\n\n# --------------------------------------------------------------------------------------\n# STAC access\n# --------------------------------------------------------------------------------------\ndef _get_json(url: str, timeout: int = 60) -> dict:\n    import requests\n    r = requests.get(url, timeout=timeout)\n    r.raise_for_status()\n    return r.json()\n\n\ndef fetch_item(scene_id: str, collection: str | None = None) -> dict:\n    """Fetch the STAC item (as a dict) for a Tanager scene.\n\n    If ``collection`` is not given we look it up in ``SCENES`` and otherwise try every\n    collection in the static catalogue until the item is found.\n    """\n    candidates = [collection] if collection else []\n    if not candidates and scene_id in SCENES:\n        candidates = [SCENES[scene_id]["collection"]]\n    candidates += [c for c in COLLECTIONS if c not in candidates]\n    last_err = None\n    for col in candidates:\n        url = f"{STAC_ROOT}/{col}/{scene_id}/{scene_id}.json"\n        try:\n            return _get_json(url)\n        except Exception as err:  # noqa: BLE001 — try the next collection\n            last_err = err\n    raise FileNotFoundError(f"Scene {scene_id} not found in any collection ({last_err})")\n\n\ndef download(url: str, dest: Path | str, chunk_mb: int = 1, verbose: bool = True) -> Path:\n    """Stream ``url`` to ``dest`` (skips the download if the file already exists)."""\n    import requests\n    dest = Path(dest)\n    if dest.exists():\n        if verbose:\n            print(f"Already downloaded: {dest}")\n        return dest\n    tmp = dest.with_suffix(dest.suffix + ".part")\n    if verbose:\n        print(f"Downloading {url}\\n  → {dest}  (Tanager HDF5 files are ~1 GB, be patient)")\n    with requests.get(url, stream=True, timeout=120) as r:\n        r.raise_for_status()\n        total = int(r.headers.get("content-length", 0))\n        done, next_report = 0, 0.0\n        with open(tmp, "wb") as f:\n            for chunk in r.iter_content(chunk_size=chunk_mb * 1024 * 1024):\n                f.write(chunk)\n                done += len(chunk)\n                if verbose and total and done / total >= next_report:\n                    print(f"  {done / 1e6:8.0f} / {total / 1e6:.0f} MB")\n                    next_report += 0.1\n    tmp.rename(dest)\n    return dest\n\n\nCATALOG_INDEX = DATA_DIR / "tanager_catalog_index.geojson"\n\n\ndef build_catalog_index(out_path=None, collections=None, max_workers: int = 8, verbose: bool = True):\n    """Crawl the static Tanager STAC catalogue and save one GeoJSON feature per scene.\n\n    Planet publishes Tanager open data as a *static* catalogue (JSON files, no search API),\n    so to search by area or date we first build our own index. Takes 1–3 minutes.\n    """\n    from concurrent.futures import ThreadPoolExecutor\n    out_path = Path(out_path) if out_path else CATALOG_INDEX\n    collections = collections or COLLECTIONS\n    links = []\n    for col in collections:\n        cj = _get_json(f"{STAC_ROOT}/{col}/collection.json")\n        links += [(col, l["href"]) for l in cj["links"] if l.get("rel") == "item"]\n    if verbose:\n        print(f"{len(links)} items in {len(collections)} collections — fetching metadata…")\n\n    def _feature(arg):\n        col, href = arg\n        it = _get_json(href)\n        p = it.get("properties", {})\n        a = it.get("assets", {})\n        return {"type": "Feature", "geometry": it.get("geometry"), "properties": {\n            "id": it["id"], "collection": col, "datetime": p.get("datetime"),\n            "location": p.get("location_description"), "cloud_percent": p.get("cloud_percent"),\n            "light_haze_percent": p.get("light_haze_percent"), "off_nadir": p.get("view:off_nadir"),\n            "sun_elevation": p.get("view:sun_elevation"), "gsd": p.get("gsd"),\n            "item_url": href, "sr_url": a.get("ortho_sr_hdf5", {}).get("href"),\n            "thumbnail": a.get("thumbnail", {}).get("href"), "visual": a.get("ortho_visual", {}).get("href"),\n        }}\n\n    with ThreadPoolExecutor(max_workers=max_workers) as ex:\n        feats = list(ex.map(_feature, links))\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    out_path.write_text(json.dumps({"type": "FeatureCollection", "features": feats}))\n    if verbose:\n        print(f"Saved {len(feats)} scenes to {out_path}")\n    return out_path\n\n\ndef load_catalog_index(refresh: bool = False):\n    """The Tanager scene index as a GeoDataFrame (builds it first if missing)."""\n    import geopandas as gpd\n    if refresh or not CATALOG_INDEX.exists():\n        build_catalog_index()\n    gdf = gpd.read_file(CATALOG_INDEX)\n    gdf["datetime"] = gdf["datetime"].astype("datetime64[ns, UTC]")\n    return gdf.sort_values("datetime").reset_index(drop=True)\n\n\ndef get_scene_path(scene_id: str, product: str = "ortho_sr_hdf5", download_if_missing: bool = True,\n                   collection: str | None = None) -> Path:\n    """Local path of a scene HDF5, downloading it from the Planet STAC catalogue if needed."""\n    p = find_local(scene_id, product)\n    if p is not None:\n        return p\n    if not download_if_missing:\n        raise FileNotFoundError(f"{scene_id} not found in {[str(d) for d in data_dirs()]}")\n    item = fetch_item(scene_id, collection)\n    url = item["assets"][product]["href"]\n    return download(url, download_dir() / f"{scene_id}_{product}.h5")\n\n\ndef get_scene(scene_id: str, **kwargs) -> "TanagerScene":\n    """Open a Tanager scene by ID (finds a local copy or downloads it)."""\n    return TanagerScene(get_scene_path(scene_id, **kwargs))\n\n\n# --------------------------------------------------------------------------------------\n# Band helpers\n# --------------------------------------------------------------------------------------\nWATER_VAPOUR_NM = ((1340.0, 1445.0), (1790.0, 1970.0))\nSWIR_EDGE_NM = 2450.0\nMIN_WL_NM = 400.0  # below ~400 nm Tanager\'s median SNR drops under 10 (see notebook 11)\n\n\ndef good_band_mask(wavelengths, file_flags=None, swir_edge: float = SWIR_EDGE_NM,\n                   min_wl: float = MIN_WL_NM, extra_bad=()) -> np.ndarray:\n    """Boolean mask of bands worth keeping.\n\n    Removes the two atmospheric water-vapour windows, the low-SNR bands below ``min_wl``,\n    the noisy detector edge beyond ``swir_edge`` nm and any ``extra_bad`` (start, stop)\n    ranges. If the per-band ``good_wavelengths`` flags from the HDF5 file are passed,\n    those are honoured too.\n    """\n    wl = np.asarray(wavelengths, dtype=float)\n    keep = (wl <= swir_edge) & (wl >= min_wl)\n    for lo, hi in tuple(WATER_VAPOUR_NM) + tuple(extra_bad):\n        keep &= ~((wl >= lo) & (wl <= hi))\n    if file_flags is not None:\n        keep &= np.asarray(file_flags).astype(bool)\n    return keep\n\n\ndef band_index(wavelengths, target_nm: float) -> int:\n    """Index of the band whose centre is closest to ``target_nm``."""\n    return int(np.argmin(np.abs(np.asarray(wavelengths) - target_nm)))\n\n\ndef nd(a, b, eps: float = 1e-6):\n    """Normalised difference (a − b) / (a + b)."""\n    return (a - b) / (a + b + eps)\n\n\n# --------------------------------------------------------------------------------------\n# The scene reader\n# --------------------------------------------------------------------------------------\nclass TanagerScene:\n    """Lightweight reader for Tanager ortho surface-reflectance HDF5 files.\n\n    Nothing is loaded into memory until you ask for it, so you can open a 1 GB file and\n    read just one band, one pixel or one small window.\n    """\n\n    def __init__(self, path):\n        self.path = Path(path)\n        self.scene_id = self.path.name[:23]\n        self.info = SCENES.get(self.scene_id, {})\n        self._f = h5py.File(self.path, "r")\n        grp = self._f["HDFEOS/GRIDS/HYP"]\n        self._fields = grp["Data Fields"]\n        sr = self._fields["surface_reflectance"]\n        self.wavelengths = np.asarray(sr.attrs["wavelengths"], dtype=float)\n        self.fwhm = np.asarray(sr.attrs["fwhm"], dtype=float)\n        self.file_good_flags = np.asarray(sr.attrs.get("good_wavelengths", np.ones(sr.shape[0])), bool)\n        self.good_bands = good_band_mask(self.wavelengths, self.file_good_flags)\n        self.shape = sr.shape  # (bands, rows, cols)\n        self._scale = float(sr.attrs.get("scale_factor", 1.0))\n        self._fill = float(np.asarray(sr.attrs.get("_FillValue", -9999.0)).ravel()[0])\n        self.epsg = int(grp.attrs["epsg_code"])\n        self.crs = f"EPSG:{self.epsg}"\n        self._parse_grid()\n\n    # ---- georeferencing -------------------------------------------------------------\n    def _parse_grid(self):\n        sm = self._f["HDFEOS INFORMATION/StructMetadata.0"][()]\n        sm = sm.decode() if isinstance(sm, bytes) else str(sm)\n        ul = re.search(r"UpperLeftPointMtrs=\\(([-\\d.]+),([-\\d.]+)\\)", sm).groups()\n        lr = re.search(r"LowerRightMtrs=\\(([-\\d.]+),([-\\d.]+)\\)", sm).groups()\n        self.ul_x, self.ul_y = float(ul[0]), float(ul[1])\n        lr_x, lr_y = float(lr[0]), float(lr[1])\n        self.res_x = (lr_x - self.ul_x) / self.shape[2]\n        self.res_y = (self.ul_y - lr_y) / self.shape[1]\n        self.bounds = (self.ul_x, lr_y, lr_x, self.ul_y)  # (left, bottom, right, top)\n\n    def transform(self, window=None, stride: int = 1):\n        """Affine geotransform (rasterio style) for the full grid or a window."""\n        from affine import Affine\n        r0, _, c0, _ = self._window(window)\n        return Affine(self.res_x * stride, 0.0, self.ul_x + c0 * self.res_x,\n                      0.0, -self.res_y * stride, self.ul_y - r0 * self.res_y)\n\n    def xy(self, row, col):\n        """Map coordinates (scene CRS, pixel centre) of a pixel."""\n        return (self.ul_x + (np.asarray(col) + 0.5) * self.res_x,\n                self.ul_y - (np.asarray(row) + 0.5) * self.res_y)\n\n    def rowcol_to_lonlat(self, row, col):\n        from rasterio.warp import transform as warp\n        x, y = self.xy(row, col)\n        lon, lat = warp(self.crs, "EPSG:4326", np.atleast_1d(x).tolist(), np.atleast_1d(y).tolist())\n        return np.array(lon), np.array(lat)\n\n    def lonlat_to_rowcol(self, lon, lat):\n        from rasterio.warp import transform as warp\n        x, y = warp("EPSG:4326", self.crs, np.atleast_1d(lon).tolist(), np.atleast_1d(lat).tolist())\n        col = ((np.array(x) - self.ul_x) / self.res_x).astype(int)\n        row = ((self.ul_y - np.array(y)) / self.res_y).astype(int)\n        return row, col\n\n    def bounds_lonlat(self):\n        """(west, south, east, north) of the grid in degrees."""\n        from rasterio.warp import transform_bounds\n        return transform_bounds(self.crs, "EPSG:4326", *self.bounds)\n\n    # ---- low level ------------------------------------------------------------------\n    def _window(self, window):\n        if window is None:\n            return 0, self.shape[1], 0, self.shape[2]\n        r0, r1, c0, c1 = window\n        return max(0, r0), min(self.shape[1], r1), max(0, c0), min(self.shape[2], c1)\n\n    def read_field(self, name: str, window=None, stride: int = 1) -> np.ndarray:\n        """Read a 2-D auxiliary field, e.g. ``\'column_water_vapour\'`` or ``\'sun_zenith\'``."""\n        r0, r1, c0, c1 = self._window(window)\n        ds = self._fields[name]\n        arr = ds[r0:r1:stride, c0:c1:stride]\n        fill = ds.attrs.get("_FillValue", None)\n        if fill is not None and arr.dtype.kind == "f":\n            arr = np.where(arr == np.asarray(fill).ravel()[0], np.nan, arr)\n        return arr\n\n    @property\n    def field_names(self):\n        return list(self._fields.keys())\n\n    # ---- masks ----------------------------------------------------------------------\n    def masks(self, window=None, stride: int = 1) -> dict:\n        """Boolean masks (True = flagged) for no-data, cloud and cirrus."""\n        out = {}\n        for key, name in (("nodata", "nodata_pixels"), ("cloud", "beta_cloud_mask"),\n                          ("cirrus", "beta_cirrus_mask")):\n            if name in self._fields:\n                m = self.read_field(name, window, stride)\n                out[key] = (m == 1) | (m == 255)\n        return out\n\n    def valid_mask(self, window=None, stride: int = 1, cloud: bool = True, cirrus: bool = True):\n        """True where a pixel is usable (not no-data, and optionally not cloud/cirrus)."""\n        m = self.masks(window, stride)\n        valid = ~m["nodata"]\n        if cloud and "cloud" in m:\n            valid &= ~m["cloud"]\n        if cirrus and "cirrus" in m:\n            valid &= ~m["cirrus"]\n        return valid\n\n    # ---- reflectance ----------------------------------------------------------------\n    def _clean(self, arr):\n        """Raw values → float32 reflectance with fill values as NaN (handles int16 subsets)."""\n        bad = arr == self._fill\n        arr = arr.astype("float32")\n        if self._scale != 1.0:\n            arr *= self._scale\n        arr[bad] = np.nan\n        return arr\n\n    def band_index(self, target_nm: float) -> int:\n        return band_index(self.wavelengths, target_nm)\n\n    def read_band(self, target_nm: float, window=None, stride: int = 1, mask: bool = True,\n                  cloud: bool = True) -> np.ndarray:\n        """Read the band closest to ``target_nm`` as a 2-D float32 array (invalid → NaN)."""\n        r0, r1, c0, c1 = self._window(window)\n        b = self.band_index(target_nm)\n        arr = self._clean(self._fields["surface_reflectance"][b, r0:r1:stride, c0:c1:stride])\n        if mask:\n            arr[~self.valid_mask(window, stride, cloud=cloud)] = np.nan\n        return arr\n\n    def read_cube(self, window=None, bands="good", stride: int = 1, mask: bool = True,\n                  cloud: bool = True, dataset: str = "surface_reflectance"):\n        """Read a spectral cube.\n\n        Parameters\n        ----------\n        window : (row_start, row_stop, col_start, col_stop) or None for the whole scene.\n        bands  : ``"good"`` (default, bad bands removed), ``"all"``, or an array of band\n                 indices / a boolean mask.\n        stride : read every n-th row and column (quick previews of the full scene).\n        mask   : set no-data / cloud / cirrus pixels to NaN.\n        dataset: ``"surface_reflectance"`` or ``"surface_reflectance_uncertainty"``.\n\n        Returns\n        -------\n        cube : float32 array (bands, rows, cols)\n        wl   : wavelengths (nm) of the returned bands\n        """\n        r0, r1, c0, c1 = self._window(window)\n        if isinstance(bands, str):\n            sel = self.good_bands if bands == "good" else np.ones(self.shape[0], bool)\n        else:\n            sel = np.asarray(bands)\n            if sel.dtype != bool:\n                tmp = np.zeros(self.shape[0], bool)\n                tmp[sel] = True\n                sel = tmp\n        # Reading all bands and subsetting in numpy is faster than h5py fancy indexing.\n        raw = self._fields[dataset][:, r0:r1:stride, c0:c1:stride]\n        cube = self._clean(raw[sel])\n        if mask:\n            cube[:, ~self.valid_mask(window, stride, cloud=cloud)] = np.nan\n        return cube, self.wavelengths[sel]\n\n    def read_pixel(self, row: int, col: int, bands="good"):\n        """Spectrum of a single pixel → (spectrum, wavelengths)."""\n        cube, wl = self.read_cube((row, row + 1, col, col + 1), bands=bands, mask=False)\n        return cube[:, 0, 0], wl\n\n    def rgb(self, window=None, stride: int = 1, wl=(640, 550, 460), p=(2, 98), mask: bool = True):\n        """True-colour (or any 3-band) composite, percentile-stretched to 0–1."""\n        bands = [self.read_band(w, window, stride, mask=mask, cloud=False) for w in wl]\n        return stretch(np.dstack(bands), p)\n\n    def to_xarray(self, window=None, bands="good", stride: int = 1, mask: bool = True):\n        """The cube as an ``xarray.DataArray`` with wavelength/y/x coordinates and a CRS."""\n        import xarray as xr\n        cube, wl = self.read_cube(window, bands, stride, mask)\n        r0, r1, c0, c1 = self._window(window)\n        rows = np.arange(r0, r1, stride)\n        cols = np.arange(c0, c1, stride)\n        x, _ = self.xy(np.zeros_like(cols), cols)\n        _, y = self.xy(rows, np.zeros_like(rows))\n        da = xr.DataArray(cube, dims=("wavelength", "y", "x"),\n                          coords={"wavelength": wl, "y": y, "x": x},\n                          name="reflectance", attrs={"scene_id": self.scene_id, "units": "reflectance"})\n        try:\n            import rioxarray  # noqa: F401 — registers the .rio accessor\n            da = da.rio.write_crs(self.crs)\n        except ImportError:\n            da.attrs["crs"] = self.crs\n        return da\n\n    def summary(self) -> str:\n        valid = self.valid_mask(stride=4)\n        m = self.masks(stride=4)\n        lines = [\n            f"Scene      : {self.scene_id}  ({self.info.get(\'place\', \'unknown place\')})",\n            f"File       : {self.path}",\n            f"Cube       : {self.shape[0]} bands × {self.shape[1]} rows × {self.shape[2]} cols",\n            f"Wavelengths: {self.wavelengths.min():.0f}–{self.wavelengths.max():.0f} nm, "\n            f"FWHM {self.fwhm.min():.1f}–{self.fwhm.max():.1f} nm",\n            f"Good bands : {self.good_bands.sum()} of {self.shape[0]}",\n            f"Grid       : {self.crs}, {self.res_x:.0f} m pixels",\n            f"No-data    : {m[\'nodata\'].mean() * 100:.1f}%   cloud flag: {m[\'cloud\'].mean() * 100:.1f}%   "\n            f"cirrus flag: {m[\'cirrus\'].mean() * 100:.1f}%   usable: {valid.mean() * 100:.1f}%",\n        ]\n        return "\\n".join(lines)\n\n    def __repr__(self):\n        return f"TanagerScene({self.scene_id}, {self.shape[1]}×{self.shape[2]} px, {self.shape[0]} bands)"\n\n    def close(self):\n        self._f.close()\n\n    def __enter__(self):\n        return self\n\n    def __exit__(self, *exc):\n        self.close()\n\n\n# --------------------------------------------------------------------------------------\n# Array helpers\n# --------------------------------------------------------------------------------------\ndef stretch(img, p=(2, 98)):\n    """Percentile stretch to 0–1 (per band for 3-D arrays shaped rows × cols × bands)."""\n    img = np.asarray(img, dtype="float32")\n    if img.ndim == 2:\n        lo, hi = np.nanpercentile(img, p)\n        return np.clip((img - lo) / (hi - lo + 1e-12), 0, 1)\n    out = np.empty_like(img)\n    for i in range(img.shape[-1]):\n        out[..., i] = stretch(img[..., i], p)\n    return out\n\n\ndef rgb_from_cube(cube, wl, bands=(640, 550, 460), p=(2, 98)):\n    """Composite from an in-memory cube (bands, rows, cols). NaN pixels become white."""\n    idx = [band_index(wl, b) for b in bands]\n    rgb = stretch(np.dstack([cube[i] for i in idx]), p)\n    rgb[np.isnan(rgb)] = 1.0\n    return rgb\n\n\ndef cube_to_pixels(cube):\n    """(bands, rows, cols) → (pixels, bands) matrix of *complete* spectra plus a 2-D mask.\n\n    Pixels with any NaN band are dropped; use ``pixels_to_image`` to put results back.\n    """\n    b, h, w = cube.shape\n    X = cube.reshape(b, -1).T\n    ok = np.all(np.isfinite(X), axis=1)\n    return X[ok], ok.reshape(h, w)\n\n\ndef pixels_to_image(values, mask, fill=np.nan):\n    """Inverse of ``cube_to_pixels``: scatter per-pixel values back onto the 2-D grid."""\n    values = np.asarray(values)\n    if values.ndim == 1:\n        out = np.full(mask.shape, fill, dtype=np.result_type(values.dtype, np.float32))\n        out[mask] = values\n        return out\n    out = np.full((values.shape[1],) + mask.shape, fill, dtype="float32")\n    out[:, mask] = values.T\n    return out\n\n\ndef plot_spectrum(ax, wl, spectrum, gap_nm: float = 20.0, **kwargs):\n    """Plot a spectrum, breaking the line where bands were removed (e.g. water vapour)."""\n    wl = np.asarray(wl, float)\n    s = np.asarray(spectrum, float)\n    breaks = np.where(np.diff(wl) > gap_nm)[0]\n    wl_p = np.insert(wl, breaks + 1, (wl[breaks] + wl[breaks + 1]) / 2)\n    s_p = np.insert(s, breaks + 1, np.nan)\n    return ax.plot(wl_p, s_p, **kwargs)\n\n\ndef shade_bad_regions(ax, color="0.85"):\n    """Grey bands over the water-vapour windows and the SWIR edge."""\n    for lo, hi in WATER_VAPOUR_NM:\n        ax.axvspan(lo, hi, color=color, zorder=0)\n    ax.axvspan(SWIR_EDGE_NM, 2510, color=color, zorder=0)\n\n\n# --------------------------------------------------------------------------------------\n# Spectral operations\n# --------------------------------------------------------------------------------------\n# Sentinel-2A MSI, approximated with Gaussian response functions (centre, FWHM in nm).\n# Real spectral response functions are published by ESA; the Gaussian version is close\n# enough to show what multispectral sensors "see".\nSENTINEL2_BANDS = {\n    "B01": (442.7, 21), "B02": (492.4, 66), "B03": (559.8, 36), "B04": (664.6, 31),\n    "B05": (704.1, 15), "B06": (740.5, 15), "B07": (782.8, 20), "B08": (832.8, 106),\n    "B8A": (864.7, 21), "B09": (945.1, 20), "B11": (1613.7, 91), "B12": (2202.4, 175),\n}\n# Landsat 8/9 OLI (approximate Gaussian).\nLANDSAT8_BANDS = {\n    "B1": (443, 16), "B2": (482, 60), "B3": (561, 57), "B4": (655, 37),\n    "B5": (865, 28), "B6": (1609, 85), "B7": (2201, 187),\n}\n# EnMAP: 224 bands, VNIR ~6.5 nm sampling (420–1000 nm) and SWIR ~10 nm (900–2450 nm).\n# Nominal grid for teaching; use the band centres from the EnMAP metadata for real work.\nENMAP_APPROX = {\n    "centers": np.concatenate([np.linspace(420, 1000, 91), np.linspace(900, 2450, 133)]),\n    "fwhm": np.concatenate([np.full(91, 8.0), np.full(133, 10.0)]),\n}\n\n\ndef resample_to_sensor(wl, spectra, centers, fwhms):\n    """Convolve hyperspectral spectra with Gaussian band responses.\n\n    ``spectra`` may be 1-D (bands,) or 2-D (n_spectra, bands). NaN bands are ignored\n    (weights are renormalised over the bands that are available).\n    """\n    wl = np.asarray(wl, float)\n    S = np.atleast_2d(np.asarray(spectra, float))\n    centers = np.asarray(centers, float)\n    sigma = np.asarray(fwhms, float) / (2 * np.sqrt(2 * np.log(2)))\n    W = np.exp(-0.5 * ((wl[None, :] - centers[:, None]) / sigma[:, None]) ** 2)  # (out, in)\n    W[W < 1e-4] = 0.0\n    finite = np.isfinite(S)\n    S0 = np.where(finite, S, 0.0)\n    num = S0 @ W.T\n    den = finite.astype(float) @ W.T\n    out = np.where(den > 0.5 * W.sum(axis=1)[None, :], num / np.maximum(den, 1e-12), np.nan)\n    return out[0] if np.ndim(spectra) == 1 else out\n\n\ndef spectral_angle(X, ref):\n    """Spectral angle (radians) between each row of ``X`` (n, bands) and ``ref`` (bands,)."""\n    X = np.atleast_2d(X)\n    ref = np.asarray(ref, float)\n    cos = (X @ ref) / (np.linalg.norm(X, axis=1) * np.linalg.norm(ref) + 1e-12)\n    return np.arccos(np.clip(cos, -1, 1))\n\n\ndef continuum_removed(wl, spectrum):\n    """Divide a spectrum by its upper convex hull (continuum removal).\n\n    Absorption features become dips below 1.0 whose depth is comparable between\n    bright and dark materials.\n    """\n    wl = np.asarray(wl, float)\n    s = np.asarray(spectrum, float)\n    ok = np.isfinite(s)\n    x, y = wl[ok], s[ok]\n    hull = [0]\n    for i in range(1, len(x)):  # monotone-chain upper hull\n        while len(hull) >= 2:\n            x1, y1 = x[hull[-2]], y[hull[-2]]\n            x2, y2 = x[hull[-1]], y[hull[-1]]\n            if (x2 - x1) * (y[i] - y1) - (y2 - y1) * (x[i] - x1) >= 0:\n                hull.pop()\n            else:\n                break\n        hull.append(i)\n    cont = np.interp(wl, x[hull], y[hull])\n    return s / cont\n\n\ndef band_depth(wl, spectrum, left, center, right):\n    """Depth of an absorption feature: 1 − R(center) / continuum(center).\n\n    ``left``/``right`` are the shoulder wavelengths; the continuum is the straight\n    line between the shoulders. Works on 1-D spectra and on cubes (bands first).\n    """\n    wl = np.asarray(wl)\n    iL, iC, iR = band_index(wl, left), band_index(wl, center), band_index(wl, right)\n    rL, rC, rR = spectrum[iL], spectrum[iC], spectrum[iR]\n    t = (wl[iC] - wl[iL]) / (wl[iR] - wl[iL])\n    cont = rL + t * (rR - rL)\n    return 1.0 - rC / (cont + 1e-12)\n\n\n# --------------------------------------------------------------------------------------\n# Export\n# --------------------------------------------------------------------------------------\ndef save_geotiff(path, array, transform, crs, nodata=None, band_names=None, cog: bool = True):\n    """Write a 2-D or 3-D (bands, rows, cols) array as a GeoTIFF (Cloud-Optimised by default)."""\n    import rasterio\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    arr = np.asarray(array)\n    if arr.ndim == 2:\n        arr = arr[None]\n    if nodata is None:\n        nodata = np.nan if arr.dtype.kind == "f" else None\n    profile = dict(count=arr.shape[0], height=arr.shape[1], width=arr.shape[2], dtype=arr.dtype,\n                   crs=crs, transform=transform, nodata=nodata, compress="deflate")\n    if cog:\n        profile.update(driver="COG", blocksize=256)\n    else:\n        profile.update(driver="GTiff", tiled=True)\n    try:\n        with rasterio.open(path, "w", **profile) as dst:\n            dst.write(arr)\n            if band_names:\n                for i, n in enumerate(band_names, start=1):\n                    dst.set_band_description(i, str(n))\n    except Exception:\n        if not cog:\n            raise\n        return save_geotiff(path, array, transform, crs, nodata, band_names, cog=False)\n    return path\n\n\n# --------------------------------------------------------------------------------------\n# Subsets (small, shareable copies of a scene window)\n# --------------------------------------------------------------------------------------\ndef write_subset(scene: TanagerScene, out_path, window, include_uncertainty: bool = False):\n    """Save a window of a scene as a small HDF5 with the same layout as the original.\n\n    Reflectance is stored as int16 × 10 000 (0.0001 precision) with gzip compression, which\n    makes a 256 × 256 window roughly 20× smaller than the source file. ``TanagerScene``\n    reads the subset exactly like a full scene.\n    """\n    r0, r1, c0, c1 = scene._window(window)\n    out_path = Path(out_path)\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    src = scene._fields\n    with h5py.File(out_path, "w") as f:\n        grp = f.create_group("HDFEOS/GRIDS/HYP")\n        for k, v in scene._f["HDFEOS/GRIDS/HYP"].attrs.items():\n            grp.attrs[k] = v\n        grp.attrs["subset_of"] = scene.path.name\n        grp.attrs["subset_window"] = np.array([r0, r1, c0, c1])\n        fields = grp.create_group("Data Fields")\n        names = ["surface_reflectance"] + (["surface_reflectance_uncertainty"] if include_uncertainty else [])\n        for name in names:\n            data = src[name][:, r0:r1, c0:c1]\n            fill = data == scene._fill\n            scaled = np.clip(np.round(data * 10000), -32767, 32767).astype("int16")\n            scaled[fill] = -32768\n            chunks = tuple(min(c, n) for c, n in zip((16, 64, 64), scaled.shape))\n            ds = fields.create_dataset(name, data=scaled, chunks=chunks, compression="gzip",\n                                       compression_opts=6, shuffle=True)\n            for k, v in src[name].attrs.items():\n                if k != "_FillValue":\n                    ds.attrs[k] = v\n            ds.attrs["scale_factor"] = 1e-4\n            ds.attrs["_FillValue"] = np.int16(-32768)\n        for name in src:\n            if src[name].ndim == 2:\n                ds = fields.create_dataset(name, data=src[name][r0:r1, c0:c1], compression="gzip")\n                for k, v in src[name].attrs.items():\n                    ds.attrs[k] = v\n        ulx = scene.ul_x + c0 * scene.res_x\n        uly = scene.ul_y - r0 * scene.res_y\n        lrx = ulx + (c1 - c0) * scene.res_x\n        lry = uly - (r1 - r0) * scene.res_y\n        sm = (f"GROUP=GridStructure\\n\\tGROUP=GRID_1\\n\\t\\tGridName=\\"HYP\\"\\n\\t\\tXDim={c1 - c0}\\n"\n              f"\\t\\tYDim={r1 - r0}\\n\\t\\tUpperLeftPointMtrs=({ulx:.2f},{uly:.2f})\\n"\n              f"\\t\\tLowerRightMtrs=({lrx:.2f},{lry:.2f})\\n\\tEND_GROUP=GRID_1\\nEND_GROUP=GridStructure\\n")\n        f.create_dataset("HDFEOS INFORMATION/StructMetadata.0", data=np.bytes_(sm))\n    return out_path\n\n\n# --------------------------------------------------------------------------------------\n# Reference labels (ESA WorldCover) aligned to a Tanager grid\n# --------------------------------------------------------------------------------------\nWORLDCOVER_CLASSES = {\n    10: ("Tree cover", "#006400"), 20: ("Shrubland", "#ffbb22"), 30: ("Grassland", "#ffff4c"),\n    40: ("Cropland", "#f096ff"), 50: ("Built-up", "#fa0000"), 60: ("Bare / sparse vegetation", "#b4b4b4"),\n    70: ("Snow and ice", "#f0f0f0"), 80: ("Permanent water", "#0064c8"), 90: ("Herbaceous wetland", "#0096a0"),\n    95: ("Mangroves", "#00cf75"), 100: ("Moss and lichen", "#fae6a0"),\n}\n\n\ndef label_path(scene_id: str) -> Path:\n    return LABEL_DIR / f"{scene_id}_worldcover2021_30m.tif"\n\n\ndef make_worldcover_labels(scene: TanagerScene, out_path=None) -> Path:\n    """Download ESA WorldCover 2021 (10 m) from Microsoft Planetary Computer and aggregate it\n    onto the Tanager 30 m grid.\n\n    The output GeoTIFF has two bands:\n      1. majority WorldCover class of the nine 10 m cells inside each 30 m pixel\n      2. purity — how many of those nine cells (0–9) agree with the majority class\n\n    Requires internet plus ``pystac-client``, ``planetary-computer`` and ``rasterio``.\n    """\n    import planetary_computer\n    import pystac_client\n    import rasterio\n    from rasterio.warp import Resampling, reproject\n\n    out_path = Path(out_path) if out_path else label_path(scene.scene_id)\n    catalog = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1",\n                                        modifier=planetary_computer.sign_inplace)\n    items = list(catalog.search(collections=["esa-worldcover"], bbox=scene.bounds_lonlat(),\n                                query={"esa_worldcover:product_version": {"eq": "2.0.0"}}).items())\n    if not items:\n        raise RuntimeError("No WorldCover tiles found for this scene")\n    from affine import Affine\n    fine_t = Affine(scene.res_x / 3, 0, scene.ul_x, 0, -scene.res_y / 3, scene.ul_y)\n    h, w = scene.shape[1] * 3, scene.shape[2] * 3\n    fine = np.zeros((h, w), dtype="uint8")\n    for it in items:\n        with rasterio.open(it.assets["map"].href) as src:\n            tmp = np.zeros((h, w), dtype="uint8")\n            reproject(rasterio.band(src, 1), tmp, dst_transform=fine_t, dst_crs=scene.crs,\n                      resampling=Resampling.nearest, src_nodata=0, dst_nodata=0)\n            fine = np.where(fine == 0, tmp, fine)\n    blocks = fine.reshape(scene.shape[1], 3, scene.shape[2], 3).transpose(0, 2, 1, 3).reshape(\n        scene.shape[1], scene.shape[2], 9)\n    classes = np.array(sorted(WORLDCOVER_CLASSES))\n    counts = np.stack([(blocks == c).sum(-1) for c in classes], axis=-1)\n    majority = classes[np.argmax(counts, axis=-1)].astype("uint8")\n    purity = counts.max(-1).astype("uint8")\n    majority[purity == 0] = 0\n    save_geotiff(out_path, np.stack([majority, purity]), scene.transform(), scene.crs, nodata=0,\n                 band_names=["worldcover_class", "purity_of_9"], cog=False)\n    return out_path\n\n\ndef load_labels(scene_id: str, window=None, min_purity: int = 0, download_if_missing: bool = False):\n    """Load the WorldCover label raster aligned to a scene → (classes, purity).\n\n    Pixels with purity below ``min_purity`` (0–9) are set to 0 (= unlabelled).\n    """\n    import rasterio\n    p = label_path(scene_id)\n    if not p.exists():\n        if not download_if_missing:\n            raise FileNotFoundError(f"{p} not found — run hu.make_worldcover_labels(scene) first")\n        make_worldcover_labels(get_scene(scene_id))\n    with rasterio.open(p) as src:\n        lab, pur = src.read(1), src.read(2)\n    if window is not None:\n        r0, r1, c0, c1 = window\n        lab, pur = lab[r0:r1, c0:c1], pur[r0:r1, c0:c1]\n    lab = lab.copy()\n    lab[pur < min_purity] = 0\n    return lab, pur\n\n\ndef label_cmap(classes=None):\n    """Matplotlib colormap, norm and legend handles for WorldCover class codes."""\n    from matplotlib.colors import BoundaryNorm, ListedColormap\n    from matplotlib.patches import Patch\n    codes = sorted(int(c) for c in (classes if classes is not None else WORLDCOVER_CLASSES) if c in WORLDCOVER_CLASSES)\n    cmap = ListedColormap([WORLDCOVER_CLASSES[c][1] for c in codes])\n    cmap.set_bad("white")\n    # One colour bin per class code: bin edges halfway between consecutive codes.\n    edges = [codes[0] - 1] + [(a + b) / 2 for a, b in zip(codes[:-1], codes[1:])] + [codes[-1] + 1]\n    norm = BoundaryNorm(edges, cmap.N)\n    handles = [Patch(color=WORLDCOVER_CLASSES[c][1], label=WORLDCOVER_CLASSES[c][0]) for c in codes]\n    return cmap, norm, handles\n\n\n# --------------------------------------------------------------------------------------\n# Spectral library\n# --------------------------------------------------------------------------------------\ndef load_library(path=None):\n    """Load the course reference library → (wavelengths, dict name → spectrum, metadata list).\n\n    The library (``data/spectral_library/tanager_reference_library.csv``) holds median\n    spectra of homogeneous areas in the course scenes, built by\n    ``tools/build_reference_library.py`` (see notebook 10). Removed bands are NaN.\n    """\n    import pandas as pd\n    path = Path(path) if path else LIBRARY_DIR / "tanager_reference_library.csv"\n    df = pd.read_csv(path)\n    wl = df["wavelength_nm"].to_numpy()\n    spectra = {c: df[c].to_numpy() for c in df.columns if c != "wavelength_nm"}\n    meta_path = LIBRARY_DIR / "tanager_reference_library.json" if path.name.startswith(\n        "tanager_reference_library") else path.with_suffix(".json")\n    meta = json.loads(meta_path.read_text()).get("entries", []) if meta_path.exists() else []\n    return wl, spectra, meta\n\n\ndef load_usgs_ascii(spectrum_file, wavelength_file):\n    """Read one USGS Spectral Library Version 7 (splib07) ASCII spectrum.\n\n    Download the ASCII archive from https://www.usgs.gov/labs/spectroscopy-lab/usgs-spectral-library\n    (e.g. ``ASCIIdata_splib07a``). Each spectrum file has a one-line header followed by\n    one reflectance value per line; the matching wavelength file (µm) sits in the same\n    archive. Deleted channels are stored as −1.23e34 and returned as NaN.\n    """\n    def _read(p):\n        vals = np.loadtxt(p, skiprows=1)\n        vals[vals < -1e30] = np.nan\n        return vals\n    wl_nm = _read(wavelength_file) * 1000.0\n    return wl_nm, _read(spectrum_file)\n\n\n# --------------------------------------------------------------------------------------\n# Misc\n# --------------------------------------------------------------------------------------\ndef attribution(scene_id: str | None = None) -> str:\n    year = scene_id[:4] if scene_id else "2025"\n    return f"Tanager STAC Data, available at www.planet.com/data/stac © {year} Planet Labs PBC. CC-BY-4.0"\n\n\ndef ensure_output_dir(sub: str = "") -> Path:\n    d = OUTPUT_DIR / sub if sub else OUTPUT_DIR\n    d.mkdir(parents=True, exist_ok=True)\n    return d'
HELPER_SHA256 = 'cdf90ac82f609e7e445bce65fbc5d220c83b7845191ed3b07d1cdd4668b3c686'
helper_dir=Path.cwd()/'demeter_helpers'
helper_dir.mkdir(exist_ok=True)
helper_path=helper_dir/'hsi_utils.py'
helper_path.write_text(HELPER_SOURCE,encoding='utf-8')
assert hashlib.sha256(helper_path.read_bytes()).hexdigest()==HELPER_SHA256
spec=importlib.util.spec_from_file_location('hsi_utils',helper_path)
hu=importlib.util.module_from_spec(spec)
sys.modules['hsi_utils']=hu
spec.loader.exec_module(hu)
sys.path.insert(0,str(helper_dir.resolve()))
print('Course helper loaded:',HELPER_SHA256)


## 1. Problem, user and decision
*(Key points from the scoping canvas.)*

- **Problem:** Prioritise vegetation inspection near Massafra, Puglia, Italy.
- **User and decision supported:** Intended agricultural/environmental analysts deciding where to inspect vegetation patterns.
- **Why hyperspectral:** NDVI 860/665 nm, NDRE 790/720 nm, NDMI 860/1650 nm and full-spectrum exploration; multispectral data can support the baseline.
- **Success criterion:** Descriptive WorldCover vegetation agreement F1 ≥ 0.80 and above the constant baseline; not a true-accuracy claim. The supplied report did not meet it.


## 2. Configuration
Scene, crop, thresholds, seeds, spectral analysis and reference-comparison parameters are defined here.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json
from pathlib import Path
from IPython.display import display
from datetime import datetime,timezone
CONFIG={
 'project':'Project_Demeter','scene_id':'20250430_101724_87_4001',
 'h5_url':'https://storage.googleapis.com/open-cogs/planet-stac/tanager1-release2-core-imagery/ortho_sr_hdf5/20250430_101724_87_4001_ortho_sr_hdf5.h5',
 'crop_size':512,'crop_center':None,'output_dir':'demeter_outputs','seed':42,
 'use_cloud_mask':True,'use_cirrus_mask':True,'require_nodata_mask':True,
 'thresholds':{'ndvi_vegetation':0.5},'sensitivity_thresholds':[.2,.3,.4,.5,.6],
 'fit_max_pixels':15000,'pca_components':10,'cluster_count':5,
 'sam_threshold_deg':10.0,'mixture_sample_pixels':5000,
 'bad_ranges_nm':[[1350,1450],[1800,2000]],
 'worldcover_validation_block_size':40,'worldcover_bootstrap_replicates':500,
 'worldcover_agreement_benchmark_f1':.80,
 'theme':'Agriculture','aoi_bbox_lonlat':None,'search_start':None,'search_end':None,
 'search_max_cloud':None,'catalog_refresh':False,
 'run_notes':'Single-date vegetation exploration; WorldCover coarse agreement, not ground-truth accuracy.'}
CONFIG.update(cache_enabled=True,cache_dir='demeter_cache',refresh_cache=False,
    refresh_spectral_cache=False,refresh_worldcover_cache=False,refresh_agreement_cache=False,strict_product_checks=True)
CONFIG.update(globals().get('CONFIG_OVERRIDE',{}))
H5_URL=CONFIG['h5_url'];CROP_SIZE=CONFIG['crop_size'];CROP_CENTER=CONFIG['crop_center']
SAM_THRESHOLD_DEG=CONFIG['sam_threshold_deg'];SECOND_DATE_URL=None
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUTPUT=Path(CONFIG['output_dir'])/RUN_ID;OUTPUT.mkdir(parents=True,exist_ok=True)
plt.rcParams.update({'figure.facecolor':'#f5f9f6','axes.titleweight':'bold','axes.spines.top':False,'axes.spines.right':False,'figure.dpi':110})
print(json.dumps(CONFIG,indent=2))


## 3. Data and quality control
Open the scene, inspect true-colour images and QA masks, and verify crop geometry. The AOI is an image crop, not an administrative boundary.


In [ ]:
import re, json, warnings, shutil
import requests, h5py, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from urllib.parse import urlparse

def numeric_vector(value):
    a = np.asarray(value)
    if a.dtype.kind in 'biuf':
        return a.astype(float).ravel()
    text = ' '.join(x.decode() if isinstance(x, bytes) else str(x) for x in a.ravel())
    return np.array([float(x) for x in re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', text)])

def load_crop(path, size=512, center=None):
    with h5py.File(path, 'r') as f:
        datasets, objects = [], [('/', f)]
        def visit(name, obj):
            objects.append((name, obj))
            if isinstance(obj, h5py.Dataset):
                datasets.append((name, obj))
        f.visititems(visit)
        inventory = []
        for name, obj in objects:
            inventory.append(f'{name} shape={getattr(obj, "shape", "group")} attributes={list(obj.attrs)}')
        (OUTPUT / 'hdf5_inventory.txt').write_text('\n'.join(inventory))
        candidates = [(n,d) for n,d in datasets if d.ndim == 3 and d.dtype.kind in 'iuf' and 'reflectance' in n.lower()]
        if not candidates:
            raise ValueError('No named 3-D reflectance dataset. Inspect hdf5_inventory.txt; do not guess a radiance cube.')
        candidates.sort(key=lambda item: ('surface_reflectance' not in item[0].lower(), item[0]))
        name, ds = candidates[0]
        metadata = []

        for owner, obj in [(name, ds)] + [(n,o) for n,o in objects if o is not ds]:
            for key, value in obj.attrs.items():
                if 'wavelength' in key.lower() and not any(s in key.lower() for s in ['good', 'bad', 'unit', 'fwhm']):
                    metadata.append((f'{owner}@{key}', value))
        metadata += [(n,d[...]) for n,d in datasets if 'wavelength' in n.lower().split('/')[-1] and not any(s in n.lower().split('/')[-1] for s in ['good','bad','fwhm','unit']) and d.size < 100000]
        selected = None
        for source, value in metadata:
            try:
                w = numeric_vector(value)
            except (TypeError, ValueError):
                continue
            for axis, length in enumerate(ds.shape):
                if w.size != length or w.size < 10 or not np.all(np.isfinite(w)):
                    continue
                wn = w * 1000 if 0.1 < np.median(w) < 10 else w
                if wn.min() > 200 and wn.max() < 4000 and np.all(np.diff(wn) > 0) and wn.min() < 480 and wn.max() > 860:
                    selected = (axis, wn, source)
                    break
            if selected:
                break
        if selected is None:
            raise ValueError('No verified wavelength vector matching a cube axis. Inspect inventory; no synthetic wavelengths will be substituted.')
        axis, wavelengths, source = selected
        spatial_axes = [a for a in range(3) if a != axis]
        rows, cols = [ds.shape[a] for a in spatial_axes]
        cr, cc = center if center is not None else (rows//2, cols//2)
        nr, nc = min(size, rows), min(size, cols)
        r0 = max(0, min(int(cr)-nr//2, rows-nr))
        c0 = max(0, min(int(cc)-nc//2, cols-nc))
        sl = [slice(None)]*3
        sl[spatial_axes[0]] = slice(r0,r0+nr)
        sl[spatial_axes[1]] = slice(c0,c0+nc)
        cube = np.moveaxis(ds[tuple(sl)].astype('float32'), axis, 0)
        for key in ['_FillValue','missing_value','nodata','no_data_value']:
            if key in ds.attrs:
                for v in numeric_vector(ds.attrs[key]):
                    cube[cube == v] = np.nan
        if 'scale_factor' in ds.attrs:
            cube *= float(np.asarray(ds.attrs['scale_factor']).squeeze())
        if 'add_offset' in ds.attrs:
            cube += float(np.asarray(ds.attrs['add_offset']).squeeze())
        cube[~np.isfinite(cube)] = np.nan
        good = np.ones(len(wavelengths),dtype=bool)
        for key in ds.attrs:
            if key.lower() == 'good_wavelengths':
                q = numeric_vector(ds.attrs[key])
                if q.size == len(good): good = q.astype(bool)
        report = dict(dataset=name, wavelength_source=source, spectral_axis=axis,
                      full_shape=list(ds.shape), crop_rows=[r0,r0+nr], crop_cols=[c0,c0+nc],
                      wavelength_range_nm=[float(wavelengths.min()),float(wavelengths.max())])
    return cube, wavelengths, good, report


In [ ]:
local = Path(urlparse(H5_URL).path).name
path = Path(local)
if not path.exists() or not h5py.is_hdf5(path):
    partial = path.with_suffix('.partial')
    with requests.get(H5_URL, stream=True, timeout=(30,180)) as response:
        response.raise_for_status()
        total = int(response.headers.get('Content-Length',0))
        received = 0
        with partial.open('wb') as out:
            for chunk in response.iter_content(8*1024*1024):
                if chunk:
                    out.write(chunk)
                    received += len(chunk)
        if total and received != total:
            raise IOError(f'Incomplete download: {received}/{total} bytes')
    if not h5py.is_hdf5(partial):
        raise ValueError('Download is not a readable HDF5 file.')
    partial.replace(path)
print('HDF5:', path, 'bytes:', path.stat().st_size)
cube, wavelengths, good, report = load_crop(path, CROP_SIZE, CROP_CENTER)
print(json.dumps(report, indent=2))

preview_indices=[int(np.argmin(abs(wavelengths-v))) for v in [665,560,480]]
preview=np.moveaxis(cube[preview_indices],0,-1).copy()
for k in range(3):
    a=preview[:,:,k];v=a[np.isfinite(a)]
    if v.size:
        lo,hi=np.percentile(v,[2,98]);preview[:,:,k]=np.clip((a-lo)/max(hi-lo,1e-6),0,1)
plt.figure(figsize=(7,6));plt.imshow(np.nan_to_num(preview,nan=0));plt.title('Analysis crop — RGB quality review');plt.axis('off');plt.show()
print('Verify geographic footprint and inspect cloud/shadow contamination before interpreting results.')


### Scene verification
Catalogue metadata and scene footprint support location review; scene cloud percentage is not crop-specific QA.


In [ ]:
scene_selection={'status':'Not checked','scene_id':CONFIG['scene_id']}
try:
    import hsi_utils as hu
    from shapely.geometry import box
    index=hu.load_catalog_index(refresh=CONFIG['catalog_refresh'])
    if index.crs is None:raise ValueError('Catalogue index lacks CRS')
    index=index.to_crs(4326)
    collections=index.groupby('id')['collection'].apply(lambda x:', '.join(sorted(set(x.astype(str)))))
    scenes=index.drop_duplicates('id').copy()
    scenes['collections']=scenes['id'].map(collections)
    scenes['datetime']=pd.to_datetime(scenes['datetime'],utc=True)
    selected=scenes[scenes.id==CONFIG['scene_id']]
    if selected.empty:raise ValueError('Configured scene ID absent from catalogue')
    row=selected.iloc[0]
    scene_selection={'status':'Selected scene found in catalogue','scene_id':str(row.id),
      'datetime':str(row.datetime),'location':str(row.get('location','')),
      'cloud_percent':None if pd.isna(row.get('cloud_percent')) else float(row.cloud_percent),
      'collections':str(row.collections),'footprint_bbox_lonlat':list(row.geometry.bounds),
      'scene_area_km2':float(selected.to_crs(6933).area.iloc[0]/1e6)}
    selected.to_file(OUTPUT/'selected_scene_footprint.geojson',driver='GeoJSON')
    candidates=scenes
    bbox=CONFIG['aoi_bbox_lonlat']
    if bbox is not None:
        if len(bbox)!=4 or bbox[0]>=bbox[2] or bbox[1]>=bbox[3]:raise ValueError('Invalid lon/lat bounding box')
        aoi=box(*bbox)
        candidates=candidates[candidates.intersects(aoi)]
        scene_selection['selected_scene_intersects_aoi']=bool(row.geometry.intersects(aoi))
        scene_selection['selected_scene_covers_aoi']=bool(row.geometry.covers(aoi))
        if not row.geometry.intersects(aoi):raise ValueError('Selected scene does not intersect supplied AOI')
    else:

        candidates=selected
        scene_selection['aoi_verification']='No verified AOI supplied; Puglia membership remains unconfirmed.'
    for key,op in [('search_start','start'),('search_end','end')]:
        if CONFIG[key]:
            date=pd.to_datetime(CONFIG[key],utc=True)
            candidates=candidates[candidates.datetime>=date] if op=='start' else candidates[candidates.datetime<=date]
    if CONFIG['search_max_cloud'] is not None:
        candidates=candidates[candidates.cloud_percent<=CONFIG['search_max_cloud']]
    show=[x for x in ['id','collections','datetime','location','cloud_percent','off_nadir'] if x in candidates]
    candidates[show].to_csv(OUTPUT/'scene_search_results.csv',index=False)
    display(candidates[show])
    fig,ax=plt.subplots(figsize=(7,5));selected.boundary.plot(ax=ax,color='#207348')
    if bbox is not None:
        import geopandas as gpd
        gpd.GeoSeries([aoi],crs=4326).boundary.plot(ax=ax,color='#bb8536')
    ax.set(xlabel='Longitude',ylabel='Latitude',title='Demeter selected-scene footprint / configured AOI');ax.grid(alpha=.2)
    fig.tight_layout();fig.savefig(OUTPUT/'scene_footprint.png',dpi=160);plt.show()
    thumb=row.get('thumbnail')
    if isinstance(thumb,str) and thumb.startswith('http'):
        from IPython.display import Image
        display(Image(url=thumb,width=600))
    print('Catalogue cloud percentage is metadata, not proof of usability; inspect image and QA.')
except Exception as error:
    scene_selection['status']=f'Catalogue verification incomplete: {type(error).__name__}: {error}'
    print(scene_selection['status'])
    if 'does not intersect supplied AOI' in str(error):raise
(OUTPUT/'scene_selection.json').write_text(json.dumps(scene_selection,indent=2,default=str))


In [ ]:
from rasterio.transform import array_bounds
from rasterio.warp import transform_bounds
from pyproj import CRS as PyCRS
from matplotlib.patches import Rectangle
with hu.TanagerScene(path) as geo_scene:
    crop_window=(*report['crop_rows'],*report['crop_cols'])
    crop_transform=geo_scene.transform(crop_window);crop_crs=geo_scene.crs
    crop_rgb=geo_scene.rgb(crop_window,wl=(665,560,480),mask=False)
    full_rgb=geo_scene.rgb(wl=(665,560,480),mask=False)
crop_h=report['crop_rows'][1]-report['crop_rows'][0];crop_w=report['crop_cols'][1]-report['crop_cols'][0];crop_bounds=array_bounds(crop_h,crop_w,crop_transform)
crop_lonlat=transform_bounds(crop_crs,'EPSG:4326',*crop_bounds,densify_pts=21)
parsed_crs=PyCRS.from_user_input(crop_crs)
if not parsed_crs.is_projected:raise ValueError('Projected CRS required for areas.')
u=parsed_crs.axis_info[0].unit_conversion_factor
pixel_area_m2=abs(crop_transform.a*crop_transform.e-crop_transform.b*crop_transform.d)*u**2
crop_details=dict(row_window=list(map(int,report['crop_rows'])),column_window=list(map(int,report['crop_cols'])),window_convention='Zero-based start inclusive; stop exclusive',shape=[crop_h,crop_w],crs=str(crop_crs),transform=list(crop_transform)[:6],pixel_size_crs_units=[abs(crop_transform.a),abs(crop_transform.e)],projected_bounds_wsen=list(crop_bounds),lonlat_bounds_wsen=list(crop_lonlat),projected_crop_area_km2=float(crop_h*crop_w*pixel_area_m2/1e6),administrative_boundary_check='Not performed; image-crop AOI, not administratively restricted')
(OUTPUT/'crop_geometry.json').write_text(json.dumps(crop_details,indent=2))
import geopandas as gpd
from shapely.geometry import box
gpd.GeoDataFrame({'scene_id':[CONFIG['scene_id']]},geometry=[box(*crop_bounds)],crs=crop_crs).to_crs(4326).to_file(OUTPUT/'cropped_AOI.geojson',driver='GeoJSON')
fig,ax=plt.subplots(figsize=(9,8));ax.imshow(full_rgb)
r0,r1=report['crop_rows'];c0,c1=report['crop_cols']
ax.add_patch(Rectangle((c0-.5,r0-.5),c1-c0,r1-r0,fill=False,edgecolor='red',linewidth=2,label='Analysis crop'))
ax.set(title='Tanager full-scene true colour | 30 April 2025',xlabel='Scene column (pixel)',ylabel='Scene row (pixel)');ax.legend()
fig.tight_layout();fig.savefig(OUTPUT/'full_scene_true_colour_crop_outline.png',dpi=170,bbox_inches='tight');plt.show();plt.close(fig)
fig,ax=plt.subplots(figsize=(8,7));ax.imshow(crop_rgb);ax.set(title='Final cropped AOI | true colour',xlabel='Crop column (pixel)',ylabel='Crop row (pixel)')
fig.tight_layout();fig.savefig(OUTPUT/'cropped_AOI_true_colour.png',dpi=170,bbox_inches='tight');plt.show();plt.close(fig)


## 4. Preprocessing
Apply good-band flags and atmospheric/noisy-band cutoffs; exclude no-data/cloud/cirrus pixels. No smoothing, continuum removal or spectral normalisation is imposed.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
SEED=CONFIG['seed']
rng=np.random.default_rng(SEED)
CORE=OUTPUT/'core_analysis';CORE.mkdir(exist_ok=True)
status={}
with hu.TanagerScene(path) as scene:
    win=(*report['crop_rows'],*report['crop_cols'])
    qa_good=hu.good_band_mask(scene.wavelengths,scene.file_good_flags,extra_bad=CONFIG['bad_ranges_nm'])
    core_cube,wl=scene.read_cube(win,bands=qa_good,mask=False)
    masks=scene.masks(win)
    if CONFIG['require_nodata_mask'] and 'nodata' not in masks:raise ValueError('Required nodata_pixels mask is missing.')
    spatial_valid=np.ones(core_cube.shape[1:],dtype=bool)
    for name,enabled in [('nodata',True),('cloud',CONFIG['use_cloud_mask']),('cirrus',CONFIG['use_cirrus_mask'])]:
        if enabled and name in masks:spatial_valid &= ~masks[name]
    core_valid=spatial_valid & np.all(np.isfinite(core_cube),axis=0)
    status['spatial_qa']={'available_masks':list(masks),'flagged_fraction':{k:float(v.mean()) for k,v in masks.items()},'helper_sha256':HELPER_SHA256,'retained_bands':len(wl),'valid_pixels':int(core_valid.sum()),'note':'Beta cloud/cirrus masks require visual inspection.'}
    fig,axes=plt.subplots(1,len(masks)+1,figsize=(5*(len(masks)+1),4),squeeze=False)
    axes=axes.ravel();axes[0].imshow(scene.rgb(win,wl=(665,560,480),mask=False));axes[0].set_title('RGB quality review')
    for ax,(name,m) in zip(axes[1:],masks.items()):ax.imshow(m,cmap='Reds',vmin=0,vmax=1);ax.set_title(name)
    for ax in axes:ax.axis('off')
    fig.tight_layout();fig.savefig(OUTPUT/'spatial_QA.png',dpi=160);plt.show()
if len(wl)<20:raise ValueError('Insufficient retained bands.')
core_cube[:,~core_valid]=np.nan
rows_v,cols_v=np.nonzero(core_valid)
Xcore=core_cube[:,core_valid].T.astype('float32')
if len(Xcore)<30:raise ValueError('Too few complete spectra.')


## 5. Features
NDVI (860/665 nm), NDRE (790/720 nm), NDMI (860/1650 nm), and full-spectrum PCA. See references below for index background; no calibrated physiological retrieval is claimed.


In [ ]:
def band(target):
    choices = np.flatnonzero(good & np.isfinite(wavelengths))
    if not len(choices): raise ValueError('No valid spectral bands.')
    i = choices[np.argmin(abs(wavelengths[choices]-target))]
    if abs(wavelengths[i]-target) > 30:
        raise ValueError(f'No suitable good band near {target} nm')
    return i

indices = {str(n):int(band(n)) for n in [480,560,665,860]}
report['selected_wavelengths_nm'] = {n:float(wavelengths[i]) for n,i in indices.items()}
red, nir = cube[band(665)], cube[band(860)]
valid = np.isfinite(red) & np.isfinite(nir) & (red >= 0) & (nir >= 0) & ((nir+red)>1e-6)
ndvi = np.full(red.shape,np.nan,dtype='float32')
np.divide(nir-red,nir+red,out=ndvi,where=valid)
rgb = np.stack([cube[band(n)] for n in [665,560,480]],axis=-1)
for k in range(3):
    vals = rgb[:,:,k]
    finite = vals[np.isfinite(vals) & (vals >= 0)]
    if not finite.size: raise ValueError('No valid RGB pixels in crop.')
    lo,hi = np.percentile(finite,[2,98])
    rgb[:,:,k] = np.clip((vals-lo)/max(hi-lo,1e-6),0,1)
rgb = np.nan_to_num(rgb,nan=0)**(1/2.2)
match = re.match(r'(\d{8})_(\d{6})',path.name)
stamp = pd.to_datetime(''.join(match.groups()),format='%Y%m%d%H%M%S').isoformat() if match else 'Unknown'
report['filename_timestamp'] = stamp
report['location_note'] = 'Puglia location supplied by user; verify scene footprint before geographic claims.'
for name,image in [('true_colour',rgb),('ndvi',ndvi)]:
    fig,ax = plt.subplots(figsize=(8,7))
    if name == 'ndvi':
        cmap = plt.get_cmap('RdYlGn').copy(); cmap.set_bad('lightgray')
        im=ax.imshow(image,cmap=cmap,vmin=-1,vmax=1)
        fig.colorbar(im,ax=ax,label='NDVI — relative vegetation greenness')
    else: ax.imshow(image)
    ax.set_title(f'Project Demeter — {name.replace("_"," ").title()}\n{stamp}')
    ax.set_axis_off(); fig.tight_layout()
    fig.savefig(OUTPUT/f'{name}.png',dpi=180,bbox_inches='tight')
    plt.show(); plt.close(fig)

visible = np.mean(np.stack([cube[band(n)] for n in [480,560,665]]),axis=0)
masks = {
'High-NDVI vegetation candidates':valid & (ndvi>0.5),
'Moderate-NDVI bright surface candidates':valid & (ndvi>0.1) & (ndvi<=0.4) & (visible>0.25),
'Low-NDVI dark surface candidates':valid & (ndvi<0) & (visible<0.2)
}
table = pd.DataFrame({'wavelength_nm':wavelengths,'good_band':good})
fig,ax=plt.subplots(figsize=(10,5))
counts={}

plot_good=good & ~((wavelengths>=1350)&(wavelengths<=1450)) & ~((wavelengths>=1800)&(wavelengths<=2000))
for label,mask in masks.items():
    counts[label]=int(mask.sum())
    if mask.sum()<50: continue
    with warnings.catch_warnings():
        warnings.simplefilter('ignore',RuntimeWarning)
        spec=np.nanmean(cube[:,mask],axis=1)
    table[label]=spec
    ax.plot(wavelengths,np.where(plot_good,spec,np.nan),label=f'{label} (n={mask.sum()})')
if len(table.columns)>2:
    ax.set(xlabel='Wavelength (nm)',ylabel='Surface reflectance',title='Project Demeter — candidate-group mean spectra')
    ax.grid(alpha=.25);ax.legend(fontsize=8);fig.tight_layout()
    fig.savefig(OUTPUT/'spectral_signatures.png',dpi=180,bbox_inches='tight');plt.show()
else:
    print('No candidate group has enough pixels; change crop. No spectra invented.')
plt.close(fig)
report['candidate_pixel_counts']=counts
report['valid_ndvi_pixels']=int(valid.sum())
report['limitations']=['Heuristic groups are not verified soil or water classes.',
'NDVI is not a diagnosis of crop health, disease or yield.',
'PNG and NumPy outputs are not georeferenced maps.',
'Cloud/shadow/glint screening requires visual review; good_wavelengths is spectral QA, not a cloud mask.']
np.save(OUTPUT/'ndvi.npy',ndvi)
np.save(OUTPUT/'wavelengths_nm.npy',wavelengths)
table.to_csv(OUTPUT/'mean_spectra.csv',index=False)
(OUTPUT/'report.json').write_text(json.dumps(report,indent=2))
(OUTPUT/'README.txt').write_text('Project Demeter\n'+ '\n'.join(report['limitations'])+'\nSee report.json for provenance and crop coordinates.')


In [ ]:
def core_band(nm):
    i=int(np.argmin(abs(wl-nm)))
    if abs(wl[i]-nm)>30:raise ValueError(f'No retained band near {nm} nm')
    return core_cube[i]
def safe_nd(a,b):
    result=np.full(a.shape,np.nan,dtype='float32')
    np.divide(a-b,a+b,out=result,where=np.isfinite(a)&np.isfinite(b)&(abs(a+b)>1e-6))
    return result
features={'NDVI':safe_nd(core_band(860),core_band(665)),
          'NDRE':safe_nd(core_band(790),core_band(720)),
          'NDMI':safe_nd(core_band(860),core_band(1650))}
ndvi=features['NDVI']
fig,axes=plt.subplots(1,3,figsize=(15,4.5))
for ax,(name,img) in zip(axes,features.items()):
    im=ax.imshow(img,cmap='RdYlGn',vmin=-.2,vmax=.9);ax.set_title(name);ax.axis('off');fig.colorbar(im,ax=ax,fraction=.04)
fig.tight_layout();fig.savefig(OUTPUT/'template_feature_maps.png',dpi=160);plt.show()


In [ ]:
import pickle,hashlib
CACHE_EVENTS=[]
def cached_compute(stage,code,names,dependencies=None):
    settings={k:v for k,v in CONFIG.items() if k not in {'output_dir','cache_dir','cache_enabled','refresh_cache','refresh_spectral_cache','refresh_worldcover_cache','refresh_agreement_cache','strict_product_checks'}}
    stat=Path(path).stat();versions={}
    for package in ['numpy','scipy','scikit-learn','rasterio']:
        try:versions[package]=importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:versions[package]='not-installed'
    signature=dict(stage=stage,code=code,config=settings,source=Path(path).name,size=stat.st_size,mtime=stat.st_mtime_ns,helper=HELPER_SHA256,versions=versions,rng=rng.bit_generator.state)
    digest=hashlib.sha256(json.dumps(signature,sort_keys=True,default=str).encode())
    for name,value in sorted((dependencies or {}).items()):
        digest.update(name.encode())
        if isinstance(value,np.ndarray):digest.update(str((value.shape,str(value.dtype))).encode());digest.update(value.tobytes(order='C'))
        else:digest.update(json.dumps(value,sort_keys=True,default=str).encode())
    directory=Path(CONFIG['cache_dir']);directory.mkdir(parents=True,exist_ok=True)
    cache_file=directory/(stage+'_'+digest.hexdigest()+'.pkl')
    group='worldcover' if stage=='worldcover_grid' else 'agreement' if stage=='agreement_blocks' else 'spectral'
    refresh=CONFIG['refresh_cache'] or CONFIG['refresh_'+group+'_cache'];hit=False
    if CONFIG['cache_enabled'] and not refresh and cache_file.exists():
        try:
            # Load only caches created locally by this notebook, never untrusted pickle files.
            with cache_file.open('rb') as stream:saved=pickle.load(stream)
            if not set(names).issubset(saved['values']):raise ValueError('Incomplete cache')
            globals().update(saved['values']);rng.bit_generator.state=saved['rng_state'];hit=True
        except Exception as error:print('Recomputing invalid cache:',stage,type(error).__name__)
    if not hit:
        exec(compile(code,'demeter_'+stage,'exec'),globals(),globals())
        if CONFIG['cache_enabled']:
            saved={'values':{name:globals()[name] for name in names},'rng_state':rng.bit_generator.state}
            temp=cache_file.with_suffix('.tmp')
            with temp.open('wb') as stream:pickle.dump(saved,stream,protocol=pickle.HIGHEST_PROTOCOL)
            temp.replace(cache_file)
    CACHE_EVENTS.append(dict(stage=stage,cache_hit=hit,key=digest.hexdigest()))
    print(stage+(': restored cache' if hit else ': computed'))


In [ ]:
def to_image(values):
    out=np.full(core_valid.shape,np.nan,dtype='float32');out[core_valid]=values;return out
cached_compute('pca_clusters', "fit=Xcore[rng.choice(len(Xcore),min(CONFIG['fit_max_pixels'],len(Xcore)),replace=False)]\npca_core=PCA(n_components=min(CONFIG['pca_components'],len(wl),len(fit)-1),svd_solver='randomized',random_state=SEED).fit(fit)\nZ=pca_core.transform(Xcore)\nkm=KMeans(n_clusters=min(CONFIG['cluster_count'],len(fit)),n_init=10,random_state=SEED).fit(pca_core.transform(fit))\ncluster=km.predict(Z)\n", ['fit','pca_core','Z','km','cluster'], {'wavelengths':wl,'valid_mask':core_valid})
np.save(CORE/'spectral_clusters.npy',to_image(cluster))


## 6. Method
Start with the fixed NDVI threshold baseline, then compare with PCA/k-means spectral clusters. SAM/SID and normalised NNLS provide cluster-prototype exploration, not verified material classification. No manual-label training or supervised classifiers are used.


In [ ]:
def sid(X, ref, eps=1e-6):

    X = np.clip(np.atleast_2d(X), eps, None)
    r = np.clip(ref, eps, None)
    P = X / X.sum(axis=1, keepdims=True)
    q = r / r.sum()
    return np.sum(P * np.log(P / q), axis=1) + np.sum(q * np.log(q / P), axis=1)
cached_compute('spectral_materials', "references=np.stack([np.median(Xcore[cluster==k],axis=0) for k in range(km.n_clusters)])\nreference_names=[f'Cluster_{k}_prototype' for k in range(km.n_clusters)]\nreference_origin='Unlabelled cluster prototypes; no material identity claimed'\nsam=np.column_stack([np.degrees(np.arccos(np.clip((Xcore@ref)/(np.linalg.norm(Xcore,axis=1)*np.linalg.norm(ref)+1e-12),-1,1))) for ref in references])\nsid_scores=np.column_stack([sid(Xcore,ref) for ref in references])\nsamclass=sam.argmin(1).astype(float);samclass[sam.min(1)>SAM_THRESHOLD_DEG]=np.nan\nfrom scipy.optimize import nnls\nsub=rng.choice(len(Xcore),min(CONFIG['mixture_sample_pixels'],len(Xcore)),replace=False)\nabundance=np.stack([nnls(references.T,Xcore[i])[0] for i in sub])\nabundance/=np.maximum(abundance.sum(1,keepdims=True),1e-12)\nresidual=np.sqrt(np.mean((Xcore[sub]-abundance@references)**2,axis=1))\n", ['references','reference_names','reference_origin','sam','sid_scores','samclass','sub','abundance','residual'], {'wavelengths':wl,'valid_mask':core_valid,'cluster':cluster})
np.savez_compressed(CORE/'material_similarity.npz',sam_min_degrees=to_image(sam.min(1)),sam_class=to_image(samclass),sid_class=to_image(sid_scores.argmin(1)),references=references,names=np.array(reference_names),wavelengths_nm=wl)
status['material_references']=reference_origin
pd.DataFrame({'row':rows_v[sub],'col':cols_v[sub],'RMSE':residual,**{name:abundance[:,i] for i,name in enumerate(reference_names)}}).to_csv(CORE/'sampled_nnls_normalised_mixtures.csv',index=False)
fig,axes=plt.subplots(1,3,figsize=(16,5))
for ax,img,title in zip(axes,[to_image(cluster),to_image(samclass),to_image(sam.min(1))],['Unlabelled spectral clusters','SAM reference index — exploratory','Minimum angle (degrees)']):
    im=ax.imshow(img,cmap='viridis');ax.set_title(title);ax.axis('off');fig.colorbar(im,ax=ax,fraction=.04)
fig.tight_layout();fig.savefig(CORE/'material_dashboard.png',dpi=160);plt.show()


In [ ]:
baseline=np.where(core_valid & np.isfinite(ndvi),(ndvi>CONFIG['thresholds']['ndvi_vegetation']).astype('float32'),np.nan)
main_name='SpectralClusters'
main_result=to_image(cluster)
result_description='Unsupervised spectral clusters; no semantic class names inferred'
fig,axes=plt.subplots(1,2,figsize=(13,5))
for ax,img,title in zip(axes,[baseline,main_result],['NDVI baseline — vegetation candidates',result_description]):
    im=ax.imshow(img,cmap='viridis');ax.set_title(title,fontsize=10);ax.axis('off');fig.colorbar(im,ax=ax,fraction=.04)
fig.tight_layout();fig.savefig(CORE/'baseline_and_main.png',dpi=160);plt.show()


## 7. Validation
Adapted to coarse agreement with ESA WorldCover 2021 v200. It is not ground truth. No manual labels are planned. Limits: temporal/seasonal mismatch, 10 m/30 m aggregation, reference errors, mixed pixels, spatial dependence and class-definition differences. Clusters have no one-to-one WorldCover mapping. Blocks describe conditional variability, not an independent train/test split.


In [ ]:
%pip install -q pystac-client planetary-computer
import pystac_client,planetary_computer,rasterio
from rasterio.warp import reproject,Resampling
from sklearn.metrics import confusion_matrix
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
crop_details.update(valid_pixels=int(core_valid.sum()),valid_area_km2=float(core_valid.sum()*pixel_area_m2/1e6))
(OUTPUT/'crop_geometry.json').write_text(json.dumps(crop_details,indent=2))
cached_compute('worldcover_grid', "cat=pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1',modifier=planetary_computer.sign_inplace)\nwc_items=sorted(cat.search(collections=['esa-worldcover'],bbox=list(crop_lonlat),query={'esa_worldcover:product_version':{'eq':'2.0.0'}}).items(),key=lambda i:i.id)\nif not wc_items:raise RuntimeError('No WorldCover 2021 v200 tiles found.')\nworldcover_labels=np.zeros(ndvi.shape,dtype='uint8')\nfor item in wc_items:\n    tile=np.zeros_like(worldcover_labels)\n    with rasterio.open(item.assets['map'].href) as source:\n        reproject(source=rasterio.band(source,1),destination=tile,src_transform=source.transform,src_crs=source.crs,src_nodata=0,dst_transform=crop_transform,dst_crs=crop_crs,dst_nodata=0,resampling=Resampling.mode)\n    fill=(worldcover_labels==0)&(tile!=0);worldcover_labels[fill]=tile[fill]\nwc_tile_ids=[item.id for item in wc_items]\n", ['worldcover_labels','wc_tile_ids'], {'transform':list(crop_transform),'crs':str(crop_crs),'shape':ndvi.shape})
wc_ok=core_valid & np.isfinite(ndvi) & (worldcover_labels!=0)
if not wc_ok.any():raise RuntimeError('No valid WorldCover overlap.')
worldcover_vegetation=np.isin(worldcover_labels,[10,20,30,40,90,95,100]);worldcover_cropland=worldcover_labels==40
wc_threshold=float(CONFIG['thresholds']['ndvi_vegetation']);wc_prediction=ndvi>wc_threshold
wc_dir=OUTPUT/'validation';wc_dir.mkdir(parents=True,exist_ok=True)
with rasterio.open(wc_dir/'worldcover_2021_aligned.tif','w',driver='GTiff',height=crop_h,width=crop_w,count=1,dtype='uint8',crs=crop_crs,transform=crop_transform,nodata=0,compress='deflate') as dst:
    dst.write(worldcover_labels,1);dst.set_band_description(1,'ESA WorldCover 2021 v200; mode aggregated')
def wc_metrics(cm):
    cm=np.asarray(cm,dtype=float);tp=np.diag(cm);ref=cm.sum(1);pred=cm.sum(0)
    div=lambda a,b:np.divide(a,b,out=np.zeros_like(a,dtype=float),where=b!=0)
    p=div(tp,pred);r=div(tp,ref);f=div(2*tp,ref+pred);j=div(tp,ref+pred-tp)
    return dict(overall_agreement=float(tp.sum()/cm.sum()),macro_F1_agreement=float(f.mean()),positive_F1_agreement=float(f[1]),positive_IoU_agreement=float(j[1])),p,r,f,j
cached_compute('agreement_blocks', "wc_rows=[];wc_classes=[];wc_intervals=[];wc_block_rows=[];wc_matrices={}\nwc_rng=np.random.default_rng(CONFIG['seed']);bs=int(CONFIG['worldcover_validation_block_size'])\nrr,cc=np.indices(ndvi.shape);ids=((rr//bs)*int(np.ceil(crop_w/bs))+cc//bs)[wc_ok];blocks=np.unique(ids)\nwc_models={'NDVI_fixed_threshold':wc_prediction,'All_vegetation_baseline':np.ones(ndvi.shape,dtype=bool)}\nfor target,reference in [('vegetation',worldcover_vegetation),('cropland',worldcover_cropland)]:\n    truth=reference[wc_ok].astype(int)\n    for method,prediction in wc_models.items():\n        pred=prediction[wc_ok].astype(int);cm=confusion_matrix(truth,pred,labels=[0,1]);wc_matrices[(target,method)]=cm\n        values,p,r,f,j=wc_metrics(cm);wc_rows.append(dict(target=target,method=method,pixels=int(cm.sum()),**values))\n        for k in [0,1]:wc_classes.append(dict(target=target,method=method,reference_class=k,reference_pixels=int(cm[k].sum()),precision_UA_wrt_WorldCover=float(p[k]),recall_PA_wrt_WorldCover=float(r[k]),F1_agreement=float(f[k]),IoU_agreement=float(j[k])))\n        pd.DataFrame(cm,index=['WorldCover negative','WorldCover positive'],columns=['Prediction negative','Prediction positive']).to_csv(wc_dir/f'{target}_{method}_agreement_confusion.csv')\n        cms=[]\n        for block in blocks:\n            sel=ids==block;bcm=confusion_matrix(truth[sel],pred[sel],labels=[0,1]);cms.append(bcm)\n            wc_block_rows.append(dict(target=target,method=method,block=int(block),pixels=int(sel.sum()),**wc_metrics(bcm)[0]))\n        if len(cms)>1:\n            cms=np.stack(cms);draws=[]\n            for repeat in range(CONFIG['worldcover_bootstrap_replicates']):draws.append(wc_metrics(cms[wc_rng.integers(0,len(cms),size=len(cms))].sum(0))[0])\n            for metric in values:\n                lo,hi=np.percentile([d[metric] for d in draws],[2.5,97.5]);wc_intervals.append(dict(target=target,method=method,metric=metric,estimate=values[metric],lower_95=float(lo),upper_95=float(hi),interpretation='Conditional scene-block variability; not true map accuracy or area CI'))\n", ['wc_rows','wc_classes','wc_intervals','wc_block_rows','wc_matrices','wc_models','blocks'], {'ndvi':ndvi,'valid_mask':core_valid,'reference':worldcover_labels})
worldcover_agreement_table=pd.DataFrame(wc_rows);wc_per_class=pd.DataFrame(wc_classes)
wc_ci=pd.DataFrame(wc_intervals,columns=['target','method','metric','estimate','lower_95','upper_95','interpretation'])
for name,df in [('worldcover_agreement_metrics',worldcover_agreement_table),('worldcover_per_class_agreement',wc_per_class),('worldcover_conditional_block_intervals',wc_ci),('worldcover_spatial_block_agreement',pd.DataFrame(wc_block_rows))]:df.to_csv(wc_dir/(name+'.csv'),index=False)
score=lambda method:float(worldcover_agreement_table.query('target=="vegetation" and method==@method').iloc[0].positive_F1_agreement)
validation_status=dict(reference_origin='ESA WorldCover 2021 v200 / Microsoft Planetary Computer',interpretation='Coarse external-reference agreement, not ground-truth accuracy',manual_labels_planned=False,sampling='All valid overlapping pixels; no reference training or threshold tuning',block_size_pixels=bs,blocks=int(len(blocks)),metrics=wc_rows,tile_ids=wc_tile_ids,resampling='mode',benchmark='Vegetation agreement F1 >= 0.80 and exceeds constant baseline; descriptive only',benchmark_met=bool(score('NDVI_fixed_threshold')>=CONFIG['worldcover_agreement_benchmark_f1'] and score('NDVI_fixed_threshold')>score('All_vegetation_baseline')))
for folder in [wc_dir,CORE]:(folder/'validation_design_status.json').write_text(json.dumps(validation_status,indent=2))
status['validation']=validation_status
display(worldcover_agreement_table);display(wc_per_class);display(wc_ci)


In [ ]:
wc_codes=[0,10,20,30,40,50,60,70,80,90,95,100]
wc_names=['No data','Tree','Shrub','Grass','Crop','Built-up','Bare/sparse','Snow/ice','Water','Wetland','Mangrove','Moss/lichen']
wc_colors=['#ddd','#006400','#ffbb22','#ffff4c','#f096ff','#fa0000','#b4b4b4','#f0f0f0','#0064c8','#0096a0','#00cf75','#fae6a0']
fig,axes=plt.subplots(1,2,figsize=(13,6));axes[0].imshow(crop_rgb);axes[0].set_title('Tanager crop RGB | 2025')
axes[1].imshow(np.searchsorted(wc_codes,worldcover_labels),cmap=ListedColormap(wc_colors),vmin=-.5,vmax=11.5);axes[1].set_title('WorldCover 2021 | 10 m mode to 30 m')
axes[1].legend(handles=[Patch(color=c,label=l) for c,l in zip(wc_colors,wc_names)],bbox_to_anchor=(1,1),loc='upper left',fontsize=8)
for ax in axes:ax.set_xlabel('Crop column (pixel)');ax.set_ylabel('Crop row (pixel)')
fig.tight_layout();fig.savefig(wc_dir/'worldcover_scene_comparison.png',dpi=170,bbox_inches='tight');plt.show();plt.close(fig)
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,target in zip(axes,['vegetation','cropland']):
    cm=wc_matrices[(target,'NDVI_fixed_threshold')];im=ax.imshow(cm,cmap='Blues')
    for (r,c),v in np.ndenumerate(cm):ax.text(c,r,f'{v:,}',ha='center',va='center')
    ax.set_xticks([0,1],['Negative','Positive']);ax.set_yticks([0,1],['Negative','Positive']);ax.set(title='WorldCover '+target+' agreement',xlabel='NDVI prediction',ylabel='WorldCover reference');fig.colorbar(im,ax=ax,label='Pixels',fraction=.04)
fig.tight_layout();fig.savefig(wc_dir/'worldcover_confusion_matrices.png',dpi=170,bbox_inches='tight');plt.show();plt.close(fig)
arr=np.full(ndvi.shape,np.nan);arr[wc_ok]=2*worldcover_vegetation[wc_ok].astype(int)+wc_prediction[wc_ok].astype(int)
fig,ax=plt.subplots(figsize=(8,7));cs=['#ddd','#e69f00','#0072b2','#009e73'];ax.imshow(np.ma.masked_invalid(arr),cmap=ListedColormap(cs),vmin=-.5,vmax=3.5)
ax.legend(handles=[Patch(color=c,label=l) for c,l in zip(cs,['Both negative','NDVI vegetation only','WorldCover vegetation only','Both vegetation'])],bbox_to_anchor=(1,1),loc='upper left');ax.set(title='Coarse vegetation agreement / disagreement',xlabel='Crop column (pixel)',ylabel='Crop row (pixel)')
fig.tight_layout();fig.savefig(wc_dir/'worldcover_agreement_map.png',dpi=170,bbox_inches='tight');plt.show();plt.close(fig)
fig,ax=plt.subplots(figsize=(8,5))
for x,method in enumerate(wc_models):
    rows=wc_ci[(wc_ci.target=='vegetation')&(wc_ci.method==method)&(wc_ci.metric=='positive_F1_agreement')]
    if len(rows):
        row=rows.iloc[0];ax.vlines(x,row.lower_95,row.upper_95,color='#207348');ax.plot(x,row.estimate,'o',color='#207348')
ax.axhline(CONFIG['worldcover_agreement_benchmark_f1'],color='gray',linestyle='--',label='Descriptive benchmark');ax.set_xticks([0,1],['Fixed NDVI','All-vegetation baseline']);ax.set(ylabel='F1 with respect to WorldCover',ylim=(0,1),title='Conditional block intervals | not true-accuracy intervals');ax.legend();fig.tight_layout();fig.savefig(wc_dir/'worldcover_baseline_uncertainty.png',dpi=170,bbox_inches='tight');plt.show();plt.close(fig)


## 8. Uncertainty and limitations
Threshold sensitivity and conditional block intervals describe different aspects of variability. Neither includes WorldCover errors or establishes true-accuracy or map-area confidence intervals. Single-date analysis, residual contamination, mixed pixels and spectral ambiguity limit decisions.


In [ ]:
sensitivity=[]
for threshold in CONFIG['sensitivity_thresholds']:
    sensitivity.append(dict(NDVI_threshold=threshold,candidate_pixels=int(np.sum(core_valid & (ndvi>threshold))),valid_pixels=int(core_valid.sum())))
pd.DataFrame(sensitivity).to_csv(CORE/'ndvi_threshold_sensitivity.csv',index=False)

centers={'B2':(490,65),'B3':(560,35),'B4':(665,30),'B5':(705,15),'B6':(740,15),'B7':(783,20),'B8A':(865,20),'B11':(1610,90),'B12':(2190,180)}
simulation={}
for name,(center,fwhm) in centers.items():
    weight=np.exp(-.5*((wl-center)/(fwhm/2.355))**2);weight[abs(wl-center)>1.5*fwhm]=0
    if weight.sum()>0:simulation[name]=(Xcore@weight)/weight.sum()
if 'B4' in simulation and 'B8A' in simulation:
    a,b=simulation['B8A'],simulation['B4'];s2ndvi=np.divide(a-b,a+b,out=np.full(len(a),np.nan),where=abs(a+b)>1e-6)
    native=ndvi[core_valid];sel=np.isfinite(native)&np.isfinite(s2ndvi)
    pd.DataFrame([dict(comparison='Approximate S2-like NDVI minus narrow-band NDVI',pixels=int(sel.sum()),bias=float(np.mean(s2ndvi[sel]-native[sel])),RMSE=float(np.sqrt(np.mean((s2ndvi[sel]-native[sel])**2))))]).to_csv(CORE/'simulated_bandpass_comparison.csv',index=False)
status['sensor_comparison']='Approximate same-scene bandpass simulation, not external validation.'
status['change_detection']='Not run: this edition supports a single-date project; a second date needs alignment, QA and coregistration before analysis.'
if SECOND_DATE_URL:print('Second date supplied but NOT processed: a separate alignment workflow is required.')
status['uncertainty']='Threshold sensitivity and conditional WorldCover-agreement block intervals; reference error and true-accuracy/area uncertainty are not quantified.'

fig,ax=plt.subplots(figsize=(7,4));ax.plot([d['NDVI_threshold'] for d in sensitivity],[d['candidate_pixels'] for d in sensitivity],'o-',color='#28784b');ax.set(xlabel='NDVI threshold',ylabel='Candidate pixels',title='Threshold sensitivity');ax.grid(alpha=.25);fig.tight_layout();fig.savefig(CORE/'threshold_sensitivity.png',dpi=160);plt.show()


## 9. Product output
Write source-georeferenced COGs with band descriptions, metadata, formal HTML report, validation tables and figures, crop geometry and results ZIP.


In [ ]:
np.save(CORE/'main_result.npy',main_result);np.save(CORE/'baseline.npy',baseline)
geo_status='Not attempted';geographic_details=None
try:
    import hsi_utils as hu
    import rasterio
    from rasterio.shutil import copy as raster_copy
    source_scene=hu.TanagerScene(path)
    window=(*report['crop_rows'],*report['crop_cols'])
    transform=source_scene.transform(window);crs=source_scene.crs
    if crs is None:raise ValueError('Missing source CRS')
    for name,img in [('result',main_result),('NDVI',ndvi),('baseline',baseline)]:
        tif=CORE/f'Project_Demeter_{name}.tif'
        hu.save_geotiff(tif,img.astype('float32'),transform,crs,band_names=[name])
        cog=CORE/f'Project_Demeter_{name}_COG.tif'
        raster_copy(str(tif),str(cog),driver='COG',compress='DEFLATE')
        with rasterio.open(cog) as ds:
            if ds.crs is None or ds.shape!=main_result.shape:raise ValueError('Export geometry check failed')
            geographic_details={'crs':str(ds.crs),'bounds':list(ds.bounds),'transform':list(ds.transform)}
    geo_status='COGs written; verify geographic alignment in GIS before submission.'
except Exception as error:
    geo_status=f'COG export incomplete: {type(error).__name__}: {error}'
    print(geo_status)
status['georeferenced_export']=geo_status
metadata={'config':CONFIG,'source_report':report,'reference_names':reference_names,'reference_origin':reference_origin,'main_method':main_name,'result_description':result_description,'status':status,'validation':validation_status,'crop_geometry':crop_details,'georeferencing':geographic_details,'python':platform.python_version(),'packages':{k:importlib.metadata.version(k) for k in ['numpy','scikit-learn','h5py','rasterio']},'attribution':'Tanager — CC-BY-4.0 © Planet Labs PBC, per supplied course materials; verify selected asset terms.','limitations':['No crop-health, disease or yield diagnosis.','Spatial QA may be incomplete; visual inspection required.','Clusters have no verified material identity.','Normalised NNLS is not exact FCLS.','No HybridSN, temporal change or physical uncertainty propagation.']}
(CORE/'Project_Demeter_metadata.json').write_text(json.dumps(metadata,indent=2,default=str))


### Reproducibility, submission checks and scalability

Cache controls are in CONFIG: `cache_enabled`, `refresh_cache`, `refresh_spectral_cache`, `refresh_worldcover_cache`, `refresh_agreement_cache`. Reuse local caches by default; set `refresh_cache=True` to recompute all stages or enable a stage-specific flag, then restore it to False. Source/configuration/code changes invalidate keys. `catalog_refresh=True` rebuilds the scene catalogue. First execution needs dependencies, internet and the source scene. Keep raw HDF5 and local caches out of submission; never load untrusted pickle files.

Country-scale monthly processing first requires confirmed scene availability and coverage, compute/storage budgets, STAC discovery and versioned scene manifests, consistent QA, windowed/tiled processing and batch queues, retries and monitoring, central COG storage and harmonised features/models across dates. Scene-specific cluster IDs are not comparable semantic classes. Monthly Tanager coverage and geographic-transfer accuracy are not established; WorldCover is not contemporaneous monthly truth.

Technical output checks do not certify clean-kernel execution, visual figure quality or independent ground-truth accuracy. Presentation and on-time submission are separate user checks.


In [ ]:
import rasterio
CHECKS={}
def product_check(name,condition,detail=''):
    CHECKS[name]={'status':'PASS' if bool(condition) else 'FAIL','detail':detail}
    if not condition and CONFIG['strict_product_checks']:raise AssertionError(name+': '+detail)
product_check('Same-shaped core outputs',ndvi.shape==baseline.shape==main_result.shape)
product_check('Nonempty valid area',bool(core_valid.any()))
product_check('Binary baseline',set(np.unique(baseline[np.isfinite(baseline)])).issubset({0.,1.}))
product_check('Finite NDVI within physical range',bool(np.all(np.abs(ndvi[np.isfinite(ndvi)])<=1.0001)))
for layer in ['NDVI','baseline','result']:
    file=CORE/f'Project_Demeter_{layer}_COG.tif';product_check(layer+' COG exists',file.is_file())
    if file.is_file():
        with rasterio.open(file) as ds:
            product_check(layer+' CRS',ds.crs==rasterio.crs.CRS.from_user_input(crop_crs))
            product_check(layer+' shape',ds.shape==ndvi.shape)
            product_check(layer+' transform',ds.transform.almost_equals(crop_transform))
            product_check(layer+' band description',bool(ds.descriptions[0]))
            product_check(layer+' COG layout',ds.tags(ns='IMAGE_STRUCTURE').get('LAYOUT')=='COG')
            expected={'NDVI':ndvi,'baseline':baseline,'result':main_result}[layer]
            product_check(layer+' array round trip',bool(np.allclose(ds.read(1),expected,equal_nan=True)))
product_check('Reference coverage',bool(wc_ok.any()))
product_check('Agreement tables present',all((wc_dir/f).is_file() for f in ['worldcover_agreement_metrics.csv','worldcover_per_class_agreement.csv','worldcover_conditional_block_intervals.csv']))
product_check('No raw HDF5 in output package',not any(f.suffix.lower() in {'.h5','.hdf5','.hdf'} for f in OUTPUT.rglob('*') if f.is_file()))
CHECKS['Clean-kernel Restart & Run All']={'status':'NOT VERIFIED HERE','detail':'Run the final file in a fresh kernel and save outputs.'}
CHECKS['Independent manually labelled spatial test']={'status':'NOT PROVIDED','detail':'WorldCover is coarse reference agreement, not ground truth.'}
CHECKS['Figures reviewed by human']={'status':'REVIEW REQUIRED','detail':'Inspect readability, titles, units and legends.'}
CHECKS['Presentation and submission deadline']={'status':'USER CHECK','detail':'Separate submission responsibilities.'}
scalability_notes={'current_scope':'One 512 x 512 Tanager crop','requirements':['Confirm monthly acquisition coverage','STAC discovery and versioned manifests','Consistent QA and windowed/tiled processing','Batch queues, retries and monitoring','Central COG storage and metadata','Harmonised features/models and transfer assessment'],'limitations':['Monthly country-wide coverage not established','Cluster IDs are not semantic labels','WorldCover is not monthly ground truth']}
(OUTPUT/'scalability_notes.json').write_text(json.dumps(scalability_notes,indent=2))
(OUTPUT/'submission_checks.json').write_text(json.dumps(CHECKS,indent=2))
(OUTPUT/'cache_events.json').write_text(json.dumps(CACHE_EVENTS,indent=2))
packages=['requests','h5py','numpy','matplotlib','pandas','scipy','scikit-learn','joblib','rasterio','rioxarray','xarray','geopandas','shapely','pyproj','pystac-client','planetary-computer']
versions={}
for package in packages:
    try:versions[package]=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:pass
(OUTPUT/'requirements_used.txt').write_text('\n'.join(k+'=='+v for k,v in versions.items())+'\n')
metadata.update(submission_checks=CHECKS,cache_events=CACHE_EVENTS,scalability=scalability_notes,packages=versions)
(CORE/'Project_Demeter_metadata.json').write_text(json.dumps(metadata,indent=2,default=str))
readme=f"""# Project Demeter
Hyperspectral Vegetation Mapping of Puglia, Italia — Space Academy 2026.

## Team contributions
| Member | Contribution |
|---|---|
| Mariolino Daniel III Julabar | Project lead and primary developer |
| AlReem AlRowahi | Project design, testing and analysis support |

## Scope and results
One Tanager crop; NDVI candidates and unlabelled spectral clusters. WorldCover 2021 v200 supplies coarse agreement, not ground-truth accuracy. No manual labels are planned. Run: {RUN_ID}. See the HTML report and validation CSVs.

## Run
Open Project_Demeter.ipynb, install dependencies and use Restart & Run All. The helper is embedded. First execution downloads the HDF5 if absent. Keep raw HDF5 and demeter_cache out of submission. CONFIG refresh_cache=True forces recomputation; restore False for reuse. Stage-specific refresh flags and country-scale requirements are documented in the notebook. requirements_used.txt records this run's versions.

## Delivery
COGs, metadata, crop geometry, validation CSVs/figures, report and ZIP. Review submission_checks.json. Presentation and submission timing remain separate tasks.

## Credits
Tanager STAC Data, available at www.planet.com/data/stac © 2025 Planet Labs PBC, CC BY 4.0.
ESA WorldCover 2021 v200, European Space Agency, CC BY 4.0; Microsoft Planetary Computer.
Supplied Space Academy 2026 helper and templates; retain their notices.
"""
(OUTPUT/'README.md').write_text(readme,encoding='utf-8')
display(pd.DataFrame([{'check':k,**v} for k,v in CHECKS.items()]))


In [ ]:
from html import escape
import base64,time
REPORT_TEMPLATE='<!DOCTYPE html>\n<html lang="en"><head><meta charset="utf-8"/><title>Project Demeter — Technical report</title><style>*{box-sizing:border-box}html{scroll-behavior:smooth}body{font-family:Arial,sans-serif;max-width:1180px;margin:0 auto;padding:28px;background:#f3f7f4;color:#253b32;line-height:1.65}h1{background:linear-gradient(120deg,#133e31,#357a57);color:white;padding:35px;border-radius:16px;font-size:40px;border-bottom:5px solid #d3ac55}h2{color:#1c563e;margin-top:40px;border-bottom:2px solid #d9e7dd;padding-bottom:10px;scroll-margin-top:20px}h3{color:#276547}.table-scroll{overflow-x:auto;border:1px solid #dce6df;border-radius:10px;margin:18px 0;background:white}table{border-collapse:collapse;width:100%;font-size:13px;margin:0}th,td{text-align:left;padding:11px;border-bottom:1px solid #e2e9e4;overflow-wrap:anywhere}th{background:#e8f0e9}tr:nth-child(even){background:#f8faf8}figure{background:white;border:1px solid #dbe6de;padding:16px;border-radius:14px;margin:25px 0}img{display:block;max-width:100%;height:auto;margin:auto}figcaption{color:#63766b;font-size:13px;padding-top:12px}nav{display:flex;flex-wrap:wrap;gap:8px;margin:18px 0}nav a{background:white;color:#236346;border:1px solid #dbe6de;border-radius:20px;padding:6px 12px;text-decoration:none;font-size:12px}.cards{display:grid;grid-template-columns:repeat(4,1fr);gap:12px}.card{background:white;border:1px solid #dbe6de;border-radius:12px;padding:18px}.card strong{display:block;color:#236346;font-size:25px}.card span{font-size:12px;color:#63766b}details{background:white;padding:14px;border-radius:10px;border:1px solid #dbe6de}summary{cursor:pointer;color:#236346}@media(max-width:750px){body{padding:15px}.cards{grid-template-columns:repeat(2,1fr)}h1{font-size:30px}}@media print{body{background:white;padding:0}nav{display:none}h1{background:white;color:#133e31}figure,table{break-inside:avoid}.table-scroll{overflow:visible}h2{break-after:avoid}}\n</style></head><body>\n<h1>Project Demeter</h1><nav><a href="#section-1">1. Executive summary</a><a href="#section-2">2. Purpose and scope</a><a href="#section-3">3. Source scene and final cropped AOI</a><a href="#section-4">4. Quality assessment</a><a href="#section-5">5. Methods</a><a href="#section-6">6. Results</a><a href="#section-7">7. Validation design: coarse external-reference agreement</a><a href="#section-8">8. Agreement results with ESA WorldCover</a><a href="#section-9">9. Uncertainty and limitations</a><a href="#section-10">10. Conclusions and next steps</a><a href="#section-11">11. Provenance and delivery</a><a href="#section-12">12. References and data credits</a></nav><div class="cards">@@CARDS@@</div><p>Hyperspectral Vegetation Mapping of Puglia, Italia<br/>Mariolino Daniel III Julabar · AlReem AlRowahi</p>\n<h2 id="section-1">1. Executive summary</h2><p>@@EXECUTIVE@@</p>\n<h2 id="section-2">2. Purpose and scope</h2><p>Agricultural and environmental analysts can use the maps to prioritise inspection. Outputs are vegetation candidates and exploratory spectral groups, not confirmed cropland, crop species, health, disease or yield.</p>\n<h2 id="section-3">3. Source scene and final cropped AOI</h2><div class="table-scroll">@@TABLE0@@</div><figure><img alt="Figure 1. Full-scene true colour with the analysis crop outlined. Display stretching may differ from the Planet thumbnail." loading="lazy" src="@@IMAGE0@@"/><figcaption>Figure 1. Full-scene true colour with the analysis crop outlined. Display stretching may differ from the Planet thumbnail.</figcaption></figure><figure><img alt="Figure 2. Final cropped AOI; bands nearest 665, 560 and 480 nm." loading="lazy" src="@@IMAGE1@@"/><figcaption>Figure 2. Final cropped AOI; bands nearest 665, 560 and 480 nm.</figcaption></figure><h3>Crop geometry</h3><div class="table-scroll">@@TABLE1@@</div><p>The geographic bounding box encloses the projected crop; it is not an administrative boundary. cropped_AOI.geojson records the transformed crop footprint.</p>\n<h2 id="section-4">4. Quality assessment</h2><p>Available no-data, cloud and cirrus flags and complete spectra define the valid area; residual contamination requires review.</p><figure><img alt="Figure 3. Available no-data, cloud and cirrus masks." loading="lazy" src="@@IMAGE2@@"/><figcaption>Figure 3. Available no-data, cloud and cirrus masks.</figcaption></figure><div class="table-scroll">@@TABLE2@@</div>\n<h2 id="section-5">5. Methods</h2><p>Scene QA → crop/band screening → indices/PCA → fixed NDVI baseline and k-means → WorldCover coarse agreement → sensitivity and delivery. Band screening excludes wavelengths below 400 nm, above 2450 nm and atmospheric water-vapour regions. Indices use bands nearest 860/665 nm (NDVI), 790/720 nm (NDRE), and 860/1650 nm (NDMI). SAM/SID and normalised NNLS use unlabelled cluster prototypes. No manual-label training or supervised classifiers are used.</p>\n<h2 id="section-6">6. Results</h2><div class="table-scroll">@@TABLE3@@</div><p>Candidate area depends on threshold and is not error-adjusted. Cluster identities remain unverified.</p><figure><img alt="Figure 4. NDVI, NDRE and NDMI maps." loading="lazy" src="@@IMAGE3@@"/><figcaption>Figure 4. NDVI, NDRE and NDMI maps.</figcaption></figure><figure><img alt="Figure 5. Heuristic candidate-group mean spectra." loading="lazy" src="@@IMAGE4@@"/><figcaption>Figure 5. Heuristic candidate-group mean spectra.</figcaption></figure><figure><img alt="Figure 6. NDVI vegetation candidates and unlabelled spectral clusters." loading="lazy" src="@@IMAGE5@@"/><figcaption>Figure 6. NDVI vegetation candidates and unlabelled spectral clusters.</figcaption></figure><figure><img alt="Figure 7. Cluster-prototype SAM diagnostics; no verified material identity." loading="lazy" src="@@IMAGE6@@"/><figcaption>Figure 7. Cluster-prototype SAM diagnostics; no verified material identity.</figcaption></figure>\n<h2 id="section-7">7. Validation design: coarse external-reference agreement</h2><p>Due to limited time and lack of manually interpreted labels, ESA WorldCover 2021 v200 is the adopted coarse reference. No manual labels are planned. Compare the fixed NDVI mask with (a) vegetation classes 10, 20, 30, 40, 90, 95, 100 and (b) cropland, class 40. Evaluate all core-valid, finite-NDVI pixels with non-zero reference coverage, including mixed pixels. WorldCover is not used for training or threshold tuning. There is no independent train/test split; blocks describe spatial variability.</p><div class="table-scroll">@@TABLE4@@</div>\n<h2 id="section-8">8. Agreement results with ESA WorldCover</h2><p>OA, precision/UA, recall/PA, F1 and IoU are with respect to WorldCover, not true land-cover accuracy. The constant all-vegetation mask is the baseline.</p><h3>8.1 Metrics and baseline comparison</h3><div class="table-scroll">@@TABLE5@@</div><h3>8.2 Per-class agreement</h3><div class="table-scroll">@@TABLE6@@</div><h3>8.3 Conditional 95% block intervals</h3><div class="table-scroll">@@TABLE7@@</div><figure><img alt="Figure 8. Tanager 2025 crop and WorldCover 2021 reference; different observation dates and class definitions." loading="lazy" src="@@IMAGE7@@"/><figcaption>Figure 8. Tanager 2025 crop and WorldCover 2021 reference; different observation dates and class definitions.</figcaption></figure><figure><img alt="Figure 9. Coarse agreement matrices. Rows: WorldCover; columns: NDVI. Not ground-truth errors." loading="lazy" src="@@IMAGE8@@"/><figcaption>Figure 9. Coarse agreement matrices. Rows: WorldCover; columns: NDVI. Not ground-truth errors.</figcaption></figure><figure><img alt="Figure 10. Reference agreement/disagreement. Disagreement is not necessarily an NDVI error." loading="lazy" src="@@IMAGE9@@"/><figcaption>Figure 10. Reference agreement/disagreement. Disagreement is not necessarily an NDVI error.</figcaption></figure><figure><img alt="Figure 11. Baseline comparison; conditional block intervals do not account for reference classification error." loading="lazy" src="@@IMAGE10@@"/><figcaption>Figure 11. Baseline comparison; conditional block intervals do not account for reference classification error.</figcaption></figure>\n<h2 id="section-9">9. Uncertainty and limitations</h2><ul><li><strong>Temporal mismatch:</strong> annual 2021 WorldCover versus Tanager on 30 April 2025; land-cover change and seasonal differences can cause disagreement.</li><li><strong>Resolution and aggregation:</strong> 10 m WorldCover is mode-aggregated to the 30 m grid; each Tanager pixel can mix several land-cover types.</li><li><strong>Different class definitions:</strong> greenness does not equal cropland; fields can be bare or fallow and vegetation can be non-cropland. Clusters have no one-to-one mapping to reference classes.</li><li><strong>Not ground truth:</strong> WorldCover is a classified product with its own errors. Agreement does not establish real land-cover accuracy.</li><li><strong>Conditional uncertainty:</strong> bootstrap intervals reflect block variability for this scene, fixed threshold and reference. Block size is provisional. They exclude reference-map error and do not establish true map-accuracy or area intervals.</li><li>Cloud, cirrus, shadow, glint and spectral ambiguity affect interpretation. Threshold sensitivity is an internal check, not a confidence interval.</li></ul><figure><img alt="Figure 12. NDVI threshold sensitivity; not a confidence interval." loading="lazy" src="@@IMAGE11@@"/><figcaption>Figure 12. NDVI threshold sensitivity; not a confidence interval.</figcaption></figure>\n<h2 id="section-10">10. Conclusions and next steps</h2><p>@@CONCLUSION@@</p>\n<h2 id="section-11">11. Provenance and delivery</h2><div class="table-scroll">@@TABLE8@@</div><details><summary>Output inventory and sizes (bytes)</summary><div class="table-scroll">@@TABLE9@@</div></details>\n<h2 id="section-12">12. References and data credits</h2><ul><li>Tanager STAC Data, available at www.planet.com/data/stac © 2025 Planet Labs PBC. CC BY 4.0.</li><li>ESA WorldCover 2021 v200, European Space Agency, CC BY 4.0; Microsoft Planetary Computer.</li><li>Rouse et al. (1974), NASA SP-351, pp. 309–317: NDVI.</li><li>USGS, Normalized Difference Moisture Index: NIR–SWIR interpretation.</li><li>Space Academy 2026 course notebooks and supplied hsi_utils.py: preprocessing, spectral analysis, validation and exports.</li></ul></body></html>'
REPORT_IMAGES=['full_scene_true_colour_crop_outline.png', 'cropped_AOI_true_colour.png', 'spatial_QA.png', 'template_feature_maps.png', 'spectral_signatures.png', 'core_analysis/baseline_and_main.png', 'core_analysis/material_dashboard.png', 'validation/worldcover_scene_comparison.png', 'validation/worldcover_confusion_matrices.png', 'validation/worldcover_agreement_map.png', 'validation/worldcover_baseline_uncertainty.png', 'core_analysis/threshold_sensitivity.png']
def kv(pairs):return pd.DataFrame(pairs,columns=['Item','Value']).to_html(index=False,escape=True)
def tab(df):return df.to_html(index=False,escape=True,float_format=lambda x:f'{x:.6f}')
vg=worldcover_agreement_table.query('target=="vegetation" and method=="NDVI_fixed_threshold"').iloc[0]
bg=worldcover_agreement_table.query('target=="vegetation" and method=="All_vegetation_baseline"').iloc[0]
cg=worldcover_agreement_table.query('target=="cropland" and method=="NDVI_fixed_threshold"').iloc[0]
benchmark='met' if validation_status['benchmark_met'] else 'not met'
executive=f'This Tanager study near Massafra produced {int(np.nansum(baseline==1)):,} vegetation-candidate pixels ({np.nansum(baseline==1)*pixel_area_m2/1e4:,.2f} ha). WorldCover vegetation agreement was {vg.overall_agreement:.6f}, with F1 {vg.positive_F1_agreement:.6f}. The descriptive benchmark was {benchmark}. This is coarse reference agreement, not ground-truth accuracy.'
conclusion=f'NDVI vegetation F1 was {vg.positive_F1_agreement:.6f} versus {bg.positive_F1_agreement:.6f} for the constant baseline; macro-F1 was {vg.macro_F1_agreement:.6f} versus {bg.macro_F1_agreement:.6f}. Different metrics reflect class imbalance and discrimination differently. Cropland F1 was {cg.positive_F1_agreement:.6f}; greenness is not cropland. The descriptive benchmark was {benchmark}, not operational approval. Review disagreements and GIS alignment; no manual annotation is planned.'
geometry_rows=[('Row window [start,stop)',crop_details['row_window']),('Column window [start,stop)',crop_details['column_window']),('Pixel indexing',crop_details['window_convention']),('Crop shape',crop_details['shape']),('CRS',crop_details['crs']),('Pixel size (CRS units)',crop_details['pixel_size_crs_units']),('Affine transform',crop_details['transform']),('Projected bounds W,S,E,N',crop_details['projected_bounds_wsen']),('Geographic bounds W,S,E,N',crop_details['lonlat_bounds_wsen']),('Crop area (km²)',crop_details['projected_crop_area_km2']),('Valid pixels',crop_details['valid_pixels']),('Valid area (km²)',crop_details['valid_area_km2']),('Administrative-boundary check',crop_details['administrative_boundary_check'])]
summary=[('Candidate pixels',int(np.nansum(baseline==1))),('Candidate area (ha)',float(np.nansum(baseline==1)*pixel_area_m2/1e4)),('Valid fraction',f'{core_valid.mean():.2%}'),('NDVI threshold',wc_threshold),('PCA components',CONFIG['pca_components']),('Spectral clusters',CONFIG['cluster_count'])]
validation_rows=[('Reference',validation_status['reference_origin']),('Resampling','Mode'),('Reference / analysis resolution','10 m / 30 m'),('Block size',validation_status['block_size_pixels']),('Blocks',validation_status['blocks']),('Descriptive benchmark',validation_status['benchmark']),('Benchmark met',validation_status['benchmark_met'])]
provenance=[('Run ID',RUN_ID),('Generated UTC',time.strftime('%Y-%m-%d %H:%M:%S',time.gmtime())),('Seed',CONFIG['seed']),('WorldCover tiles',validation_status['tile_ids']),('Python',platform.python_version()),('NumPy',np.__version__)]
files=[(str(f.relative_to(OUTPUT)),f.stat().st_size) for f in sorted(OUTPUT.rglob('*')) if f.is_file() and f.name!='Project_Demeter_Report.html']
report_tables=[kv([('Scene',CONFIG['scene_id']),('Acquisition','30 April 2025, 10:17:24.870 UTC'),('Source','Planet Tanager agriculture'),('Reference','ESA WorldCover 2021 v200')]),kv(geometry_rows),kv([('COG export status',geo_status)]),kv(summary),kv(validation_rows),tab(worldcover_agreement_table),tab(wc_per_class),tab(wc_ci.drop(columns=['interpretation'],errors='ignore')),kv(provenance),kv(files)]
html=REPORT_TEMPLATE
for i,table in enumerate(report_tables):html=html.replace(f'@@TABLE{i}@@',table)
for i,relative in enumerate(REPORT_IMAGES):
    image=OUTPUT/relative
    if not image.is_file():raise FileNotFoundError(f'Report figure missing: {relative}')
    html=html.replace(f'@@IMAGE{i}@@','data:image/png;base64,'+base64.b64encode(image.read_bytes()).decode('ascii'))
cards=''.join(f'<div class="card"><strong>{escape(value)}</strong><span>{escape(label)}</span></div>' for value,label in [(f'{int(core_valid.sum()):,}','Valid analysed pixels'),(f'{np.nansum(baseline==1)*pixel_area_m2/1e4:,.2f} ha','NDVI candidate area'),(f'{vg.positive_F1_agreement:.4f}','Vegetation F1 vs WorldCover'),(benchmark.upper(),'Descriptive benchmark')])
html=html.replace('@@CARDS@@',cards).replace('@@EXECUTIVE@@',escape(executive)).replace('@@CONCLUSION@@',escape(conclusion))
if '@@' in html:raise RuntimeError('Unresolved report placeholder.')
import re
extra_html=('<h3>Scalability: monthly country-scale processing</h3><p>Country-wide monthly coverage is not established. Confirm acquisition coverage and budgets, automate STAC discovery and QA, use tiled/windowed processing and batch queues with retries and monitoring, and store versioned COGs centrally. Harmonise scene/date features and models; cluster IDs are not semantic classes. WorldCover is not monthly ground truth.</p><h3>Technical output and submission checks</h3>'+pd.DataFrame([{'Check':k,**v} for k,v in CHECKS.items()]).to_html(index=False,escape=True))
html=re.sub(r'(?=<h2[^>]*>11\.)',lambda match:extra_html,html,count=1)
report_path=OUTPUT/'Project_Demeter_Report.html';report_path.write_text(html,encoding='utf-8')
metadata.update(validation=validation_status,crop_geometry=crop_details,reference_attribution='ESA WorldCover 2021 v200, European Space Agency, CC BY 4.0')
(CORE/'Project_Demeter_metadata.json').write_text(json.dumps(metadata,indent=2,default=str))
print('Wrote formal report:',report_path)


In [ ]:
product_check('Formal HTML report generated',(OUTPUT/'Project_Demeter_Report.html').is_file())
product_check('Contribution README generated',(OUTPUT/'README.md').is_file())
(OUTPUT/'submission_checks.json').write_text(json.dumps(CHECKS,indent=2))
manifest=[]
for file in sorted(OUTPUT.rglob('*')):
    if file.is_file() and file.name!='manifest.json':
        checksum=hashlib.sha256()
        with file.open('rb') as stream:
            for chunk in iter(lambda:stream.read(1024*1024),b''):checksum.update(chunk)
        manifest.append({'file':str(file.relative_to(OUTPUT)),'bytes':file.stat().st_size,'sha256':checksum.hexdigest()})
(OUTPUT/'manifest.json').write_text(json.dumps(manifest,indent=2))
archive=shutil.make_archive(CONFIG['project']+'_results','zip',OUTPUT)
import zipfile
with zipfile.ZipFile(archive) as package:
    if any(Path(name).suffix.lower() in {'.h5','.hdf5','.hdf'} for name in package.namelist()):raise AssertionError('Raw HDF5 found in ZIP')
print('Final results package:',archive)


In [ ]:
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Results package:',archive)


## 10. Conclusions and next steps

- **Answer to the problem:** Project Demeter provides a georeferenced NDVI vegetation-candidate map and five exploratory spectral clusters for the 512 × 512 crop near Massafra, Puglia, Italy. The latest supplied report, run `20261113T101551_705207Z` (generated 2026-11-13 10:17:26 UTC), records 145,771 candidates covering 13,119.39 ha. WorldCover supplies a coarse external agreement check, not ground-truth accuracy. The descriptive agreement benchmark was not met: NDVI vegetation F1 is below 0.80 and below the constant all-vegetation baseline's positive F1. NDVI's higher macro-F1 indicates better balance between positive and negative reference classes, not a general accuracy advantage.

- **Key numbers:** Transcribed from the latest supplied executed report, not recomputed by this final documentation edit.

| Item | Value / status |
|---|---|
| Source scene / acquisition | `20250430_101724_87_4001` / 30 April 2025, 10:17:24.870 UTC |
| Report run / generation UTC | `20261113T101551_705207Z` / 2026-11-13 10:17:26 |
| Crop / pixel size / CRS | 512 × 512 / 30 × 30 m / EPSG:32633 |
| Crop rows / columns | [127,639) / [182,694), zero-based, stop exclusive |
| Projected bounds W,S,E,N (m) | 674520, 4486140, 689880, 4501500 |
| Geographic bounds W,S,E,N (degrees) | 17.059788602, 40.504259480, 17.245570198, 40.645918944 |
| Crop / valid analysed area | 235.9296 / 231.2397 km² |
| Valid pixels / valid fraction | 256,933 / 98.01% |
| NDVI candidates / candidate area | 145,771 / 13,119.39 ha |
| NDVI threshold / PCA components / spectral clusters | 0.5 / 10 / 5 |
| WorldCover reference / tile | ESA WorldCover 2021 v200 / `ESA_WorldCover_10m_2021_v200_N39E015` |
| Comparison pixels / blocks / block size | 256,933 / 169 / 40 × 40 pixels |
| Vegetation overall agreement / positive F1 / IoU | 0.696676 / 0.787899 / 0.650027 |
| Vegetation precision / recall w.r.t. WorldCover | 0.993010 / 0.653016 |
| Conditional 95% vegetation-F1 block interval | 0.765342–0.809086; not a true-accuracy interval |
| All-vegetation baseline overall agreement / positive F1 | 0.862742 / 0.926314 |
| NDVI / constant baseline macro-F1 | 0.627832 / 0.463157 |
| Cropland overall agreement / positive F1 / IoU | 0.430315 / 0.273993 / 0.158744 |
| Descriptive benchmark | Not met; not an operational accuracy requirement |
| Technical output checks | 28 PASS in supplied report |
| Administrative-boundary check | Not performed; image-crop AOI, not administratively restricted |
| Clean-kernel execution / human figure review | Not verified / review required |

- **What the user gets:** NDVI, baseline and spectral-cluster COGs; processing metadata; crop geometry JSON and GeoJSON; WorldCover agreement and per-class CSVs; confusion matrices; conditional block intervals; full-scene and crop true-colour images; reference comparison and disagreement maps; a standalone styled HTML report; contribution README; dependency-version record; cache-event log; scalability notes; submission checks; and a results ZIP. The supplied report documents successful COG export and passing CRS, transform, band-description, array round-trip and no-raw-HDF5 checks. The report is a prototype, not a deployed operational service.

- **Next steps:** Run this final edited notebook with Restart & Run All in a fresh kernel and save outputs; inspect geographic alignment, residual QA and figure readability; use measured disagreements and limitations in the presentation. No manual labels are planned. WorldCover's own errors, 2021/2025 temporal and seasonal mismatch, 10 m/30 m aggregation, mixed pixels, spatial dependence and class-definition differences limit interpretation. Greenness is not cropland and clusters have no one-to-one reference-class mapping. Conditional intervals exclude reference errors and do not establish true map-accuracy or map-area confidence intervals. Country-scale monthly coverage and transfer accuracy are not established; scalability requirements are documented rather than claimed as implemented national coverage.

## References and data credits

- Tanager STAC Data, available at [www.planet.com/data/stac](https://www.planet.com/data/stac) © 2025 Planet Labs PBC (CC-BY-4.0).
- ESA WorldCover 2021 v200 (CC-BY 4.0), accessed through Microsoft Planetary Computer.
- Rouse et al. (1974), NASA SP-351, pp. 309–317: NDVI background.
- USGS, Normalized Difference Moisture Index: NIR–SWIR interpretation.
- Supplied Space Academy 2026 project template and course notebooks: structure, preprocessing, spectral analysis, validation and reporting.
